# KíLín - Code-Project Evaluation Notebook

This standalone notebook evaluates a development checkout as both the
assistant implementation and the source corpus: only `kilin/` and
`tutorials/` are copied into a fresh workspace. Allowed suffixes are `.py`,
`.ipynb`, `.m`, `.md`, `.json`, `.js`, and `.html`. They use normal source
processing; no special parser exceptions are made. A suffix without files
in the checkout is **not exercised**, not automatically passed.

Run the notebook sequentially, one section at a time. Optional long tools,
file reading, persistence and model comparisons are off initially. Each
case keeps its native trace-derived summary, full raw answer, original
model outputs and repository-relative evidence paths. All model and code
quality results depend on this run's sources and settings, recorded in the
manifest; the repository is not pinned to a release.

Saved outputs below are historical and may cite tutorial filenames from
before the reordering. The executable cells use the current names; rerun
the notebook to obtain comparable results for the new source snapshot.

## Interpretation

- **Routing**: `MATCH`/`REVIEW` tests the declared intent, tool and links.
- **Execution**: `completed`, `partial`, `incomplete`, `cancelled`, `failed`,
  `skipped` or `probed`. An invocation is not proof of completion.
- **Scope/structure**: source paths, definition spans and selected AST/graph
  contracts. Static relationships are not an exhaustive runtime graph.
- **Quality**: independently checked facts and manual answer assessment.
  Keywords, valid citations and compiled code are not behavioral proofs.

`---` denotes no expected route, no tool invoked, or an inapplicable check.
Model-setup failures and missing prerequisites produce explicit skipped
comparison records. Disabled optional sections remain visibly untested in
coverage. Full responses in expandable notes are never truncated. Generated
Python is parsed and compiled but **never executed**.

The configured model IDs and switching API use **MLX**, not PyTorch. Query
latency is recorded separately from model startup. Independent comparison
tasks receive fresh histories; continuity tests deliberately use each
model's own prior answers. One trial per model is exploratory, not a robust
statistical benchmark. No external evaluator model is called.

## 1. Experiment configuration

Run from the current KíLín development checkout. This experiment copies
source folders into its own independent workspace and records the Git HEAD
and working-tree status where available. Source hashes are authoritative
even when the checkout is dirty or not managed by Git.

By default, core scenarios and lightweight probes run; expensive tools,
exhaustive analysis and model comparisons are opt-in. Adjust flags before
executing the setup cell. All model IDs must already be installed locally
when `ALLOW_DOWNLOAD=False`.

Generation settings are explicit and saved in the workspace, then reapplied
per session/case. Intent and tool thinking policies may intentionally
change a particular call; inspect native traces for effective per-call
values. Both requested and active models, the embedding model, router list,
profile hash and startup time are exported. Avoid changing several settings
at once when comparing models.

In [1]:
from datetime import datetime
from pathlib import Path
import subprocess

EXPERIMENT_ID = datetime.now().strftime("%Y%m%d-%H%M%S")
PROJECT_NAME = f"trial_code_routing_{EXPERIMENT_ID}"
ALLOW_DOWNLOAD = False

MODEL_IDS = {
    "qwen8b": "Qwen/Qwen3-8B-MLX-4bit",
    "qwen14b": "Qwen/Qwen3-14B-MLX-4bit",
    "qwen38_27b": "lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly",
}
PRIMARY_MODEL = "qwen8b"
ROUTER_MODEL = None  # Inherit the project router.
COMPARISON_MODELS = ("qwen8b", "qwen38_27b")
COMPARISON_ROUTERS = (
    "Qwen/Qwen3-4B-MLX-4bit",
    "Qwen/Qwen3-8B-MLX-4bit",
)

CODE_EXTENSIONS = (
    ".py", ".ipynb", ".m", ".md", ".json", ".js", ".html"
)
BACKEND = "mlx"  # These model IDs and runtime switching use MLX.
GENERATION_SETTINGS = {
    "temperature": 0.0,
    "thinking": True,
    "max_new_tokens": 8192,
}
RETRIEVAL_COUNT = 10
RUN_PERSISTENCE_CHECK = True
RUN_LONG_TOOLS = True
RUN_FULL_SOURCE_READ = True
RUN_CODE_REVIEW = True
RUN_MODEL_COMPARISON = True
RUN_ROUTER_COMPARISON = True
RUN_GUI_EXPORT_TEST = True

if BACKEND != "mlx":
    raise ValueError("This notebook configures MLX models, not PyTorch.")
assert PRIMARY_MODEL in MODEL_IDS
assert all(key in MODEL_IDS for key in COMPARISON_MODELS)

repo_root = Path.cwd().resolve()
while not (
    (repo_root / "pyproject.toml").is_file()
    and (repo_root / "kilin/project_types/code.json").is_file()
    and (repo_root / "tutorials").is_dir()
):
    if repo_root == repo_root.parent:
        raise RuntimeError("Run inside the KíLín development repository.")
    repo_root = repo_root.parent
project_dir = repo_root / "sandbox" / PROJECT_NAME


def _git_info(arguments):
    """Read optional source-version details without changing the checkout.

    Parameters
    ----------
    arguments : list of str
        Arguments passed to the Git executable.

    Returns
    -------
    str or None
        Standard output, or None when Git cannot provide a result.

    """
    try:
        result = subprocess.run(
            ["git", "-C", str(repo_root), *arguments],
            capture_output=True, text=True, timeout=15, check=False,
        )
    except (OSError, subprocess.TimeoutExpired):
        return None
    return result.stdout.strip() if result.returncode == 0 else None


SOURCE_VERSION = {
    "git_commit": _git_info(["rev-parse", "HEAD"]),
    "git_status": _git_info([
        "status", "--porcelain", "--untracked-files=normal",
        "--", "kilin", "tutorials",
    ]),
}
print("Development repository:", repo_root)
print("Experiment:", PROJECT_NAME)
print("Destination:", project_dir)
print("Git commit:", SOURCE_VERSION["git_commit"] or "unavailable")
print(
    "Source tree modified:",
    "unavailable" if SOURCE_VERSION["git_status"] is None
    else bool(SOURCE_VERSION["git_status"]),
)
print("Extensions:", ", ".join(CODE_EXTENSIONS))
print("Answering model:", MODEL_IDS[PRIMARY_MODEL])
print("Router: project settings")


Development repository: /Users/giovannivolpe/Documents/GitHub/KiLin
Experiment: trial_code_routing_20261109-161523
Destination: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_code_routing_20261109-161523
Git commit: e87dcc2bd01b5a973da89b08c2053b07c347900f
Source tree modified: False
Extensions: .py, .ipynb, .m, .md, .json, .js, .html
Answering model: Qwen/Qwen3-8B-MLX-4bit
Router: project settings


## 2. Recording and evaluation helpers

The helpers use `RAGSession` and KíLín's native router. They record actual
answers, selected tools, previous-turn links, model calls, and trace-authored
turn representations. They do not change KíLín's retrieval or generation
logic. Explicit expected **source paths** disambiguate same-named files
such as `__init__.py`.

A routing decision is scored separately from source scope, execution,
trace consistency and content anchors. Full answers remain available
unchanged in the expanded details. Code correctness requires checking
actual definitions and relationships; there is no model-as-judge loop.

Coverage counts distinguish full attempts, correct operation selection,
completion, cancellation and skips. A cancelled test can match its routing
and cancellation contracts but never counts as completed semantic search.
Unreported metadata remains unknown rather than zero.

In [2]:
import ast
import csv
import re
import html
import json
import sys
from hashlib import sha256
from contextlib import contextmanager
from time import perf_counter

from IPython.display import HTML, display

import kilin

ACTIVE_PACKAGE_ROOT = Path(kilin.__file__).resolve().parents[1]
if ACTIVE_PACKAGE_ROOT != repo_root:
    raise RuntimeError(
        "The imported kilin package is not the development "
        "checkout. Start the kernel from this checkout. "
        f"Loaded {ACTIVE_PACKAGE_ROOT}; expected {repo_root}."
    )

from kilin.answering import route_question, strip_thinking
from kilin.conversations import ConversationStore
from kilin.project_type import project_type
from kilin.rag import RAGSession

CASE_RESULTS = []
CASE_INDEX = {}
CONVERSATION_INDEX = {}
STARTUP_RECORDS = []
INFRASTRUCTURE_RESULTS = {}


def code_intents():
    """Return configured code intents without hard-coded tool names.

    Returns
    -------
    dict
        Current name-to-configuration mapping from code.json.

    """
    path = repo_root / "kilin" / "project_types" / "code.json"
    with path.open(encoding="utf-8") as stream:
        return json.load(stream)["intents"]


def expected_tool(intent):
    """Return the tool configured for an intent, or None for model answers.

    Parameters
    ----------
    intent : str or None
        Intent identifier in the current code project type.

    Returns
    -------
    str or None
        Configured tool name, if present.

    """
    tool = code_intents().get(intent, {}).get("tool") or {}
    return tool.get("name")


def _source_names(evidence):
    """Identify filenames in saved source evidence, retaining order.

    Parameters
    ----------
    evidence : sequence
        Evidence objects persisted by RAGSession.

    Returns
    -------
    list of str
        Unique document basenames.

    """
    found = []
    for item in evidence:
        path = getattr(item, "path", None)
        if path is None and isinstance(item, dict):
            path = item.get("path") or item.get("source")
        if path:
            name = Path(str(path)).name
            if name not in found:
                found.append(name)
    return found


def _trace_operation(stages, stage_name, suffix):
    """Locate an operation by semantic suffix, not sequence number.

    Parameters
    ----------
    stages : dict
        One native trace turn's stages.
    stage_name : str
        Stage name such as ``01_question``.
    suffix : str
        End of the operation name, without its sequence prefix.

    Returns
    -------
    dict
        Matching operation or an empty mapping.

    """
    operations = stages.get(stage_name, {}).get("operations", {})
    return next(
        (value for key, value in operations.items() if key.endswith(suffix)),
        {},
    )


def _trace_turns(path):
    """Load committed turns from the native conversation trace.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace JSON path.

    Returns
    -------
    dict
        Native turns keyed by their recorded one-based IDs.

    """
    try:
        with path.open(encoding="utf-8") as stream:
            trace = json.load(stream)
        turns = trace["sessions"]["05_conversations"]["turns"]
        return turns
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return {}


def _recorded_representation(turn, traced):
    """Read the KíLín-created summary, preferring its trace event.

    Parameters
    ----------
    turn : ConversationTurn
        Committed turn containing KíLín's saved representation.
    traced : dict or None
        Representation from ``conversation.turn.representation``.

    Returns
    -------
    dict
        Recorded summary, provenance, and trace/checkpoint agreement.

    Notes
    -----
    Neither source is reconstructed from the raw answer. If both recorded
    representations exist, they should agree on their semantic fields.

    """
    traced = traced if isinstance(traced, dict) else {}
    parameters = getattr(turn, "parameters", {}) or {}
    checkpoint = parameters.get("representation") or {}
    if not isinstance(checkpoint, dict):
        checkpoint = {}
    trace_ok = isinstance(traced.get("summary"), str)
    saved_ok = isinstance(checkpoint.get("summary"), str)
    if trace_ok:
        representation = traced
        origin = "trace"
    elif saved_ok:
        representation = checkpoint
        origin = "checkpoint"
    else:
        representation = {}
        origin = "unavailable"
    consistent = None
    if trace_ok and saved_ok:
        fields = ("question", "summary", "sources")
        consistent = all(
            traced.get(field) == checkpoint.get(field)
            for field in fields
        )
    return {
        "summary": representation.get("summary", ""),
        "sources": list(representation.get("sources") or []),
        "origin": origin,
        "consistent": consistent,
    }


def _native_route(path, turn_number):
    """Read routing and timing details from one saved native trace turn.

    Parameters
    ----------
    path : pathlib.Path
        Conversation trace path.
    turn_number : int
        One-based turn number in this conversation.

    Returns
    -------
    dict
        Router, history, retrieval, and model-call diagnostics.

    """
    turns = _trace_turns(path)
    recorded = turns.get(f"turn_{turn_number:04d}", {})
    if not recorded:
        return {"trace_available": False}
    stages = recorded.get("stages", {})

    routed = _trace_operation(
        stages, "01_question", "intent_classification_decided"
    ).get("details", {})
    history = _trace_operation(
        stages, "01_question", "history_selection_result"
    ).get("details", {})
    retrieval = _trace_operation(
        stages, "03_retrieval", "retrieval_result"
    ).get("details", {})
    settings = _trace_operation(
        stages, "01_question", "turn_settings_snapshot"
    ).get("details", {})
    representation = _trace_operation(
        stages, "01_question", "conversation_turn_representation"
    ).get("details", {})
    model_calls = []
    for stage in stages.values():
        for name, value in stage.get("operations", {}).items():
            if name.endswith("llm_call_output"):
                model_calls.append(value.get("details", {}))
    return {
        "trace_available": True,
        "classification": routed.get("classification", {}),
        "shortcut": routed.get("shortcut"),
        "fallback": routed.get("fallback"),
        "dropped_turns": routed.get("dropped_turns", []),
        "router_raw": routed.get("raw"),
        "history": history,
        "retrieval": retrieval,
        "settings": settings,
        "model_calls": model_calls,
        "representation": representation,
    }

def _source_paths(evidence):
    """Return distinct saved source paths relative to the source root.

    Parameters
    ----------
    evidence : sequence
        Saved evidence objects or their checkpoint dictionary forms.

    Returns
    -------
    list of str
        Normalized paths, with distinct directories preserved.

    """
    found = []
    for item in evidence:
        path = getattr(item, "path", None)
        if path is None and isinstance(item, dict):
            path = item.get("path") or item.get("source")
        if not path:
            continue
        value = str(path).replace("\\", "/")
        prefix = str(project_dir / "sources").replace("\\", "/") + "/"
        if value.startswith(prefix):
            value = value[len(prefix):]
        elif value.startswith("sources/"):
            value = value[len("sources/"):]
        if value not in found:
            found.append(value)
    return found


def _has_source(paths, target):
    """Match an expected repository-relative path without basename loss.

    Parameters
    ----------
    paths : sequence of str
        Saved source paths.
    target : str
        Expected path, such as ``kilin/rag.py``.

    Returns
    -------
    bool
        Whether the saved evidence includes that source path.

    """
    value = target.replace("\\", "/").strip("/")
    return any(
        path == value or path.endswith("/" + value)
        for path in paths
    )


/Users/giovannivolpe/Documents/GitHub/KiLin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def _gui_callbacks(events, choice="default", confirmation="quick"):
    """Create non-networked callbacks and record each scripted decision.

    Parameters
    ----------
    events : list
        Mutable list populated with GUI-equivalent interactions.
    choice : str or tuple, optional
        ``default``, ``first20``, ``save_all``, ``cancel``, or a tuple
        ``("document", filename)``. A value of ``none`` omits the adapter.
    confirmation : str, optional
        ``quick``, ``continue``, or ``cancel`` for model plans. Network plans
        are always cancelled.

    Returns
    -------
    tuple
        Choice callback and confirmation callback for ``RAGSession.ask``.

    """

    def choose(question, options):
        selected = None
        if isinstance(choice, tuple) and choice[0] == "document":
            target = choice[1].casefold()
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if (
                        item.split(" (", 1)[0].casefold() == target
                        or item.casefold().endswith("/" + target)
                    )
                ),
                None,
            )
        elif choice in ("default", "first20"):
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item == "Show the first 20 here"
                ),
                None,
            )
            if selected is None and choice == "default" and options:
                if "which paper" in question.casefold() or (
                    "which document" in question.casefold()
                ):
                    selected = 0
        elif choice == "save_all":
            selected = next(
                (
                    i
                    for i, item in enumerate(options)
                    if item.startswith("Save all ")
                    and item.endswith(" to outputs/ as Markdown")
                ),
                None,
            )
        events.append(
            {
                "kind": "choice",
                "question": question,
                "options": list(options),
                "selected": selected,
            }
        )
        return selected

    def confirm(plan):
        decision = (
            "cancel"
            if getattr(plan, "operation", "model") == "network"
            else confirmation
        )
        events.append(
            {
                "kind": "confirmation",
                "tool": plan.tool,
                "operation": getattr(plan, "operation", "model"),
                "summary": plan.summary,
                "items": plan.items,
                "batches": plan.batches_to_run,
                "quick_items": plan.quick_items,
                "selected": decision,
            }
        )
        return decision

    choice_callback = None if choice == "none" else choose
    return choice_callback, confirm




CONVERSATION_SECTIONS = {
    "empty-project": "3.1",
    "direct-routes": "5",
    "code-relations": "6.1",
    "code-continuity": "6.2",
    "results-tools": "7",
    "gui-paths": "7.1",
    "routing-boundaries": "8",
    "hard-scope": "9",
    "code-ground-truth": "9.1",
    "optional-review": "10",
    "optional-file-review": "10",
    "code-generation": "11",
    "answer-models-seed": "12",
    "router-seed": "13",
    "reopen-check": "5.1",
    "symbol-boundaries": "9.2",
}


def _conversation_name(label):
    """Generate a persistent name using the section and recorded cases.

    Parameters
    ----------
    label : str
        Logical test group identifier.

    Returns
    -------
    str
        Notebook section and first/last recorded case IDs.

    """
    ids = [
        record["id"] for record in CASE_RESULTS
        if record["label"] == label
    ]
    if not ids:
        return label
    section = CONVERSATION_SECTIONS.get(label)
    if label.startswith("answer-model-"):
        section = "12"
    elif label.startswith("router-model-"):
        section = "13"
    span = ids[0] if len(ids) == 1 else f"{ids[0]}-{ids[-1]}"
    return f"{section}. {span}" if section else f"{label}: {span}"


@contextmanager
def conversation(
    label, *, model=None, router=None, branch_from=None,
    resume_from=None, optional=False,
):
    """Open a session without obscuring initialization or body failures.

    Parameters
    ----------
    label : str
        Unique group identifier for this evaluation.
    model : str or None, optional
        Answering model key in MODEL_IDS.
    router : str or None, optional
        Router model identifier.
    branch_from : str or None, optional
        Saved conversation ID to clone with independent state.
    resume_from : str or None, optional
        Saved ID to reopen, without changing its name or adding a turn.
    optional : bool, optional
        On setup failure, record the error and yield None so comparison
        cases can be explicitly skipped. Body failures still propagate.

    Yields
    ------
    dict or None
        Active context, or None for a failed optional initialization.

    Notes
    -----
    Close initialized sessions even when switching models fails. Never
    rename or index an unregistered conversation. Cleanup errors cannot
    replace an earlier exception. Startup time is separate from case time.

    """
    selected = model or PRIMARY_MODEL
    if selected not in MODEL_IDS:
        raise ValueError(f"Unknown model key: {selected}")
    if label in CONVERSATION_INDEX:
        raise ValueError(f"Conversation label already used: {label}")
    if branch_from and resume_from:
        raise ValueError("Choose branching or reopening, not both.")
    store = ConversationStore(project_dir)
    if branch_from or resume_from:
        store.ensure()
    session = None
    registered = False
    started = perf_counter()
    setup = {
        "label": label, "model": MODEL_IDS[selected],
        "router": router or ROUTER_MODEL, "status": "initializing",
    }
    STARTUP_RECORDS.append(setup)
    try:
        try:
            if resume_from:
                info = next(
                    item for item in store.conversations()
                    if item.conversation_id == resume_from
                )
            else:
                info = store.create(branch_from=branch_from)
            path = store.state_path(info)
            session = RAGSession(
                project_dir, conversation_path=path,
                router_model=router,
            )
            if session.backend != BACKEND:
                raise RuntimeError("This notebook requires the MLX backend.")
            session.use_chat_model(MODEL_IDS[selected])
            session.chat_session.configure_generation(GENERATION_SETTINGS)
        except Exception as error:
            setup.update(status="failed", error=f"{type(error).__name__}: "
                         f"{error}", seconds=perf_counter() - started)
            if not optional:
                raise
            print(f"Conversation setup failed: {label}: {setup['error']}")
            yield None
            return
        setup.update(status="ready", seconds=perf_counter() - started)
        effective = _effective_settings(session)
        context = {
            "label": label, "store": store, "info": info,
            "path": path, "session": session, "case_turn_ids": {},
            "model": selected, "router": router or ROUTER_MODEL,
            "parent": branch_from, "effective_settings": effective,
        }
        CONVERSATION_INDEX[label] = {
            "path": str(path.relative_to(project_dir)),
            "conversation_id": info.conversation_id,
            "model": selected, "router": router or ROUTER_MODEL,
            "parent": branch_from, "reopened": resume_from,
            "effective_settings": effective,
            "startup_seconds": setup["seconds"],
        }
        registered = True
        print(f"Conversation {label}: {path.name}")
        print("  Answering:", session.chat_session.model_id)
        print("  Router:", session.router_model)
        yield context
    finally:
        earlier_error = sys.exception()
        if session is not None:
            try:
                session.close()
            except BaseException as cleanup_error:
                setup["cleanup_error"] = repr(cleanup_error)
                if earlier_error is None:
                    raise
                earlier_error.add_note(f"Cleanup failed: {cleanup_error!r}")
        if registered and not resume_from:
            try:
                renamed = store.rename(
                    info.conversation_id, _conversation_name(label)
                )
                CONVERSATION_INDEX[label]["name"] = renamed.name
            except Exception as rename_error:
                if earlier_error is None:
                    raise
                earlier_error.add_note(f"Rename failed: {rename_error!r}")


def _effective_settings(session):
    """Record active model configuration independently of project defaults.

    Parameters
    ----------
    session : RAGSession
        Initialized session, before a case or after a saved settings update.

    Returns
    -------
    dict
        Actual backend, model IDs, generation and index metadata.

    """
    chat = session.chat_session
    return {
        "backend": session.backend,
        "answering_model": chat.model_id,
        "router_model": session.router_model,
        "embedding_model": session.project.profile.embedding_model(
            session.backend
        ),
        "generation": {
            "temperature": chat.temperature,
            "thinking": chat.thinking,
            "max_new_tokens": chat.max_new_tokens,
        },
        "index_fingerprint": session.indexes.fingerprint,
        "index_status": session.index_status,
    }


def _expected_links(context, case_ids):
    """Resolve named earlier cases to one-based conversation turn IDs.

    Parameters
    ----------
    context : dict
        Active conversation context.
    case_ids : tuple
        Previously run case identifiers.

    Returns
    -------
    list of int
        One-based expected turn numbers.

    """
    missing = [
        case_id
        for case_id in case_ids
        if case_id not in context["case_turn_ids"]
    ]
    if missing:
        raise ValueError(f"Missing earlier cases: {missing}")
    return [context["case_turn_ids"][item] for item in case_ids]


def _execution_outcome(record):
    """Separate completion of an attempted operation from routing accuracy.

    Parameters
    ----------
    record : dict
        Observed case, including native tool status and callback decisions.

    Returns
    -------
    str
        skipped, failed, probed, partial, incomplete, cancelled or completed.
        Completed does not imply scientific or behavioral correctness.

    """
    if record.get("status") == "SKIPPED":
        return "skipped"
    if record.get("status") == "ERROR":
        return "failed"
    if record["mode"] == "route-only":
        return "probed"
    completion = record.get("completion") or {}
    status = record.get("tool_status") or completion.get("status")
    if status in ("partial", "incomplete", "failed"):
        return status
    if completion or record.get("finish_reason") == "length":
        return "incomplete"
    events = record.get("gui_events", [])
    if status in ("cancelled", "canceled") or any(
        event.get("selected") == "cancel"
        or (event.get("kind") == "choice"
            and event.get("selected") is None)
        for event in events
    ):
        return "cancelled"
    if not strip_thinking(record.get("answer") or "").strip():
        return "incomplete"
    return "completed"


def _check_generated_function(record):
    """Parse and compile the selected Python block without executing it.

    Parameters
    ----------
    record : dict
        Expected function, optional parameter/call checks, and raw answer.
        Receives generated_check metadata identifying the selected block.

    Returns
    -------
    list of str
        Compilation, signature, structural and formatting warnings. Passing
        these checks is not evidence of correct runtime behavior.

    """
    expected = record.get("expected_generated_function")
    if not expected:
        return []
    answer = strip_thinking(record.get("answer") or "")
    blocks = re.findall(r"```([^\n]*)\n([\s\S]*?)```", answer)
    candidates = []
    errors = []
    for number, (language, source) in enumerate(blocks, 1):
        if language.strip().casefold() not in ("", "py", "python"):
            continue
        try:
            tree = ast.parse(source)
        except SyntaxError as error:
            errors.append(f"block {number}: {error.msg}")
            continue
        definitions = [
            item for item in tree.body
            if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef))
            and item.name == expected
        ]
        if definitions:
            candidates.append((number, source, tree, definitions))
    record["generated_check"] = {
        "candidate_blocks": [item[0] for item in candidates],
        "executed": False, "behavior": "not evaluated",
    }
    if len(candidates) != 1 or len(candidates[0][3]) != 1:
        return [
            "expected one unambiguous Python block defining " + expected
        ] + errors
    number, source, tree, definitions = candidates[0]
    check = record["generated_check"]
    check.update(block_number=number, compiles=False)
    try:
        compile(source, "<generated Python>", "exec", dont_inherit=True)
    except (SyntaxError, ValueError, TypeError) as error:
        return [f"generated Python cannot compile: {error}"]
    check["compiles"] = True
    node = definitions[0]
    problems = []
    parameters = record.get("expected_generated_parameters")
    actual = [arg.arg for arg in node.args.posonlyargs + node.args.args]
    check["parameters"] = actual
    if parameters is not None and (
        actual != parameters or node.args.vararg or node.args.kwarg
        or node.args.kwonlyargs or node.args.defaults
        or isinstance(node, ast.AsyncFunctionDef)
    ):
        problems.append("generated function signature differs")
    calls = {ast.unparse(item.func) for item in ast.walk(node)
             if isinstance(item, ast.Call)}
    for call in record.get("expected_generated_calls", []):
        if call not in calls:
            problems.append(f"missing expected API call: {call}")
    if any(len(line) > 79 for line in source.splitlines()):
        problems.append("generated Python exceeds 79 columns")
    return problems


def _check_record(record):
    """Evaluate routing, source scope, and diagnostics independently.

    Parameters
    ----------
    record : dict
        Completed routing probe or committed conversation turn.

    Returns
    -------
    dict
        Separate lists of routing, scope, execution, trace, and content
        concerns. Content anchors are not code correctness proofs.

    """
    checks = {
        "routing": [],
        "scope": [],
        "execution": [],
        "trace": [],
        "content": [],
    }
    route = checks["routing"]
    expected = record.get("expected_intent")
    if expected is not None and record.get("intent") != expected:
        route.append("intent")
    if record["mode"] == "full" and expected is not None:
        if record.get("tool") != record.get("expected_tool"):
            route.append("tool")
    standalone = record.get("expected_standalone")
    if standalone is not None and record.get("standalone") != standalone:
        route.append("standalone")
    actual = set(record.get("related_turn_ids", []))
    required = set(record.get("expected_related_ids", []))
    forbidden = set(record.get("forbidden_related_ids", []))
    if not required.issubset(actual):
        route.append("missing prior-turn link")
    if forbidden & actual:
        route.append("unrelated prior-turn link")
    if record.get("expected_retrieval") is not None:
        if record.get("retrieval_mode") != record["expected_retrieval"]:
            route.append("retrieval policy")
    if record.get("intent") in record.get("disallowed_intents", []):
        route.append("disallowed intent")

    if record["mode"] == "full":
        sources = set(record.get("sources", []))
        if not set(record.get("expected_sources", [])).issubset(sources):
            checks["scope"].append("missing source in retrieved evidence")
        if set(record.get("forbidden_sources", [])) & sources:
            checks["scope"].append("out-of-scope retrieved source")
        paths = record.get("source_paths", [])
        for name in record.get("expected_paths", []):
            if not _has_source(paths, name):
                checks["scope"].append(f"missing source path: {name}")
        for name in record.get("forbidden_paths", []):
            if _has_source(paths, name):
                checks["scope"].append(f"unexpected source: {name}")
        for fact in record.get("expected_definitions", []):
            if not any(
                _has_source([item.get("path", "")], fact["file"])
                and (item.get("start_line") or 0) <= fact["line"]
                <= (item.get("end_line") or -1)
                for item in record.get("evidence_spans", [])
            ):
                checks["scope"].append(
                    f"definition not in evidence: {fact['symbol']}"
                )
        checks["content"].extend(_check_generated_function(record))
        if record.get("outcome") != record.get(
            "expected_outcome", "completed"
        ):
            checks["execution"].append(
                f"expected {record.get('expected_outcome', 'completed')}; "
                f"observed {record.get('outcome')}"
            )
        if record.get("invalid_citations"):
            checks["execution"].append("citation warning (check quotes)")
        if record.get("completion"):
            checks["execution"].append("incomplete answer")
        if not record.get("trace", {}).get("trace_available"):
            checks["trace"].append("native trace unavailable")
        if record.get("summary_source") != "trace":
            checks["trace"].append("summary not from trace event")
        if record.get("summary_consistent") is False:
            checks["trace"].append("trace/checkpoint summary mismatch")
        logged_model = record.get("trace", {}).get(
            "settings", {}
        ).get("chat_model")
        if logged_model and logged_model != record.get("model"):
            checks["trace"].append("active-model trace mismatch")
        for prior in record.get("previous", []):
            if prior.get("summary_consistent") is False:
                checks["trace"].append("prior summary mismatch")
                break
        answer = strip_thinking(record.get("answer") or "").casefold()
        for term in record.get("expected_answer_terms", []):
            if term.casefold() not in answer:
                checks["content"].append(f"missing answer anchor: {term}")
    return checks


def _grade_record(record):
    """Attach independent diagnostic fields and a routing-only verdict.

    Parameters
    ----------
    record : dict
        Populated case record, updated in place.

    """
    record["outcome"] = _execution_outcome(record)
    expected = record.get("expected_intent")
    record["operation_selected"] = bool(expected) and (
        record.get("intent") == expected
        and (record["mode"] == "route-only"
             or record.get("tool") == record.get("expected_tool"))
    )
    checks = _check_record(record)
    record["checks"] = checks
    record["issues"] = [
        f"{kind}: {message}"
        for kind, problems in checks.items()
        for message in problems
    ]
    record["status"] = (
        "REVIEW"
        if checks["routing"]
        else "MATCH" if record.get("expected_intent") else "---"
    )
    record["scope_status"] = (
        "REVIEW" if checks["scope"]
        else "OK" if record.get("expected_sources")
        or record.get("forbidden_sources")
        or record.get("expected_paths")
        or record.get("forbidden_paths") else "---"
    )


In [4]:
def _brief(value, limit=95):
    """Return short escaped text for a compact diagnostic table.

    Parameters
    ----------
    value : object
        Value to show.
    limit : int, optional
        Maximum characters before truncation.

    Returns
    -------
    str
        Escaped, whitespace-normalized label.

    """
    text = " ".join(str(value if value is not None else "-").split())
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return html.escape(text)


def _final_answer_text(answer):
    """Remove reasoning for content checks, not answer summaries.

    Parameters
    ----------
    answer : str or None
        Raw answer, potentially containing model reasoning.

    Returns
    -------
    str
        Answer text excluding thinking blocks.

    """
    return strip_thinking(answer or "")


def show_case(record):
    """Show one compact table and optional full answer diagnostics.

    Parameters
    ----------
    record : dict
        Recorded routing-only or end-to-end test case.

    """
    expected = record.get("expected_intent") or "---"
    actual = record.get("intent") or "-"
    tool = record.get("tool") or "---"
    related = record.get("related_turn_ids", [])
    need = record.get("expected_related_ids", [])
    tokens = record.get("usage", {})
    token_text = (
        f"{tokens.get('input_tokens', 0)}/" f"{tokens.get('output_tokens', 0)}"
        if tokens
        else "-"
    )
    source_names = record.get("sources", [])
    source_text = f"{len(source_names)}: " + ", ".join(source_names)
    scanned = record.get("files_scanned")
    if scanned is not None:
        source_text += f" ({scanned} scanned)"
    status = record["status"]
    checks = record.get("checks", {})
    other = ", ".join(
        name for name in ("scope", "execution", "trace", "content")
        if checks.get(name)
    ) or "---"
    body = "".join(
        "<td style='padding:4px 7px;border-bottom:1px solid #bbb'>"
        + value
        + "</td>"
        for value in (
            _brief(f"{expected} → {actual}", 55),
            _brief(tool, 22),
            _brief(f"need {need}; got {related}", 29),
            _brief(record.get("retrieval_mode") or "---", 12),
            _brief(source_text, 47),
            _brief(token_text, 22),
            _brief(f"{record.get('seconds', 0):.2f}s"),
            _brief(status, 18),
            _brief(record.get("outcome") or "---", 15),
            _brief(other, 36),
        )
    )
    title = (
        f"<strong>{_brief(record['id'])}</strong> · "
        f"{_brief(record['question'], 145)}"
    )
    header = "".join(
        f"<th style='padding:4px 7px;text-align:left'>"
        f"{html.escape(name)}</th>"
        for name in (
            "Expected → actual",
            "Tool",
            "Prior links",
            "Retrieval",
            "Evidence",
            "Tokens in/out",
            "Time",
            "Routing check",
            "Execution",
            "Other checks",
        )
    )
    answer = record.get("answer") or ""
    summary = record.get("answer_summary") or ""
    if record["mode"] == "route-only":
        summary = "---"
    elif not summary and record.get("summary_source") == "unavailable":
        summary = "(summary unavailable)"
    elif not summary:
        summary = "(no final-answer summary)"
    one_line_answer = _brief(summary, 180)
    issues = "<div>" + "</div><div>".join(
        f"<b>{html.escape(kind)}:</b> "
        f"{_brief(', '.join(messages), 300)}"
        for kind, messages in checks.items() if messages
    ) + "</div>" if any(checks.values()) else "none"
    origin = f"{record.get('label', '?')} / {record.get('model', '?')}"
    trace_path = record.get("trace_path", "-")
    route = record.get("trace", {})
    model_calls = route.get("model_calls", [])
    generation_events = record.get("generation_events")
    tool_attempts = record.get("tool_internal_attempts")
    tool_text = (
        str(tool_attempts) if tool_attempts is not None
        else "not reported" if record.get("tool") else "---"
    )
    limit_hits = sum(
        call.get("finish_reason") == "length" for call in model_calls
    )
    prior_turns = record.get("previous", [])
    chosen = set(record.get("related_turn_ids", []))
    prior_html = "".join(
        f"<li>{'*' if item['turn'] in chosen else ' '} "
        f"#{item['turn']} {_brief(item['question'], 100)}: "
        f"{_brief(item['summary'], 120)} "
        f"({_brief(item.get('summary_source'), 15)})</li>"
        for item in prior_turns
    )
    refers_back = route.get("classification", {}).get(
        "refers_back", route.get("refers_back")
    )
    scope_note = ""
    if scanned is not None:
        scope_note = (
            f"<div><b>Scan:</b> {scanned} files scanned; "
            f"{record.get('files_with_mentions')} with matches; "
            f"{record.get('listed')} items listed; "
            f"{len(source_names)} source files in saved evidence.</div>"
        )
    elif record.get("passages_read") is not None:
        scope_note = (
            "<div><b>Passages actually read:</b> "
            f"{record['passages_read']} "
            f"(planned scope: {record.get('planned_passages')})."
            "</div>"
        )
    details = (
        f"<div><b>Scenario:</b> {_brief(origin)} · "
        f"<b>Turn:</b> {_brief(record.get('turn_number'))} · "
        f"<b>Trace:</b> {_brief(trace_path, 130)}</div>"
        f"<div><b>Checks:</b> {issues}</div>"
        f"<div><b>Skip/setup reason:</b> "
        f"{_brief(record.get('skip_reason') or '---', 600)}</div>"
        f"<div><b>Generated-code checks:</b> "
        f"{_brief(record.get('generated_check') or '---', 600)}</div>"
        f"<div><b>Summary provenance:</b> "
        f"{_brief(record.get('summary_source'))} · "
        f"<b>Trace/checkpoint agree:</b> "
        f"{_brief(record.get('summary_consistent'))}</div>"
        f"<div><b>Configured tool:</b> "
        f"{_brief(expected_tool(record.get('intent')) or '---')} · "
        f"<b>Evidence filenames:</b> {_brief(source_names, 350)}</div>"
        f"<div><b>Exact source paths:</b> "
        f"{_brief(record.get('source_paths', []), 800)}</div>"
        f"{scope_note}"
        f"<div><b>Shortcut:</b> {_brief(route.get('shortcut'))} · "
        f"<b>Fallback:</b> {_brief(route.get('fallback'))}</div>"
        f"<div><b>History IDs:</b> "
        f"{_brief(record.get('history_turn_ids', []))} · "
        f"<b>Refers back:</b> {_brief(refers_back)}</div>"
        f"<div><b>Generation-stage events:</b> "
        f"{generation_events if generation_events is not None else '---'}"
        f" · <b>Tool-internal attempts:</b> {_brief(tool_text)} · "
        f"<b>Length stops (generation stage):</b> {limit_hits} · "
        f"<b>Finish:</b> {_brief(record.get('finish_reason'))}</div>"
        "<details><summary>Earlier answer snapshots "
        "(* means linked by router)</summary><ul>"
        f"{prior_html}</ul></details>"
        "<div><b>Complete saved response:</b></div>"
        "<pre style='white-space:pre-wrap;max-height:300px;"
        "overflow:auto;font-size:12px'>"
        f"{html.escape(answer)}</pre>"
    )
    display(
        HTML(
            f"<div style='margin:8px 0 15px 0;font-size:12px'>"
            f"<div style='margin-bottom:5px'>{title}</div>"
            "<div style='overflow-x:auto'>"
            "<table style='border-collapse:collapse;width:100%;"
            "font-size:11px'><thead><tr>"
            f"{header}</tr></thead><tbody><tr>{body}</tr></tbody></table>"
            "</div>"
            f"<div style='margin:3px 0'><b>Recorded summary:</b> "
            f"{one_line_answer}</div>"
            f"<details style='margin-top:5px'><summary>Answer and routing "
            f"details</summary>{details}</details></div>"
        )
    )


def inspect_case(case_id, *, show_answer=True, show_prior=True):
    """Print the complete saved answer, related context, and trace data.

    Parameters
    ----------
    case_id : str
        ID shown by the compact diagnostic table.
    show_answer : bool, optional
        Whether to print the full answer, including any reasoning tags.
    show_prior : bool, optional
        Whether to print earlier linked questions and saved answer summaries.

    Returns
    -------
    dict
        Unmodified record collected by the evaluation harness.

    """
    record = CASE_INDEX[case_id]
    print(
        json.dumps(
            {
                key: value
                for key, value in record.items()
                if key not in ("answer", "trace")
            },
            indent=2,
            ensure_ascii=False,
        )
    )
    if show_prior:
        print("\nPREVIOUS TURN SNAPSHOT")
        for item in record.get("previous", []):
            marker = (
                "*"
                if item["turn"] in record.get("related_turn_ids", [])
                else " "
            )
            print(
                f"{marker} #{item['turn']}: {item['question']}\n"
                f"    {item['summary']}"
            )
    if show_answer:
        print("\nFULL ANSWER\n", record.get("answer", ""))
    print("\nNATIVE ROUTING DIAGNOSTICS")
    print(json.dumps(record.get("trace", {}), indent=2, ensure_ascii=False))
    return record


In [5]:
def _previous_snapshot(context):
    """Capture KíLín's recorded prior-turn summaries for the router.

    Parameters
    ----------
    context : dict
        Active conversation state.

    Returns
    -------
    list of dict
        Prior-turn metadata with authoritative summary provenance.

    Notes
    -----
    Read ``conversation.turn.representation`` events first. When a trace
    is unavailable, use the same representation saved in turn parameters.
    Do not independently summarize the raw assistant response.

    """
    snapshot = []
    native_turns = _trace_turns(context["path"])
    for index, turn in enumerate(context["session"].turns, start=1):
        stages = native_turns.get(
            f"turn_{index:04d}", {}
        ).get("stages", {})
        traced = _trace_operation(
            stages, "01_question", "conversation_turn_representation"
        ).get("details", {})
        representation = _recorded_representation(turn, traced)
        snapshot.append(
            {
                "turn": index,
                "question": turn.question,
                "summary": representation["summary"],
                "summary_source": representation["origin"],
                "summary_consistent": representation["consistent"],
                "summary_sources": representation["sources"],
                "sources": _source_names(turn.evidence),
                "intent": turn.parameters.get("intent"),
                "tool": (turn.parameters.get("tool") or {}).get("name"),
            }
        )
    return snapshot


def _register(record):
    """Retain and display one result without changing any test outcome.

    Parameters
    ----------
    record : dict
        New unique case record.

    """
    if record["id"] in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {record['id']}")
    CASE_RESULTS.append(record)
    CASE_INDEX[record["id"]] = record
    show_case(record)


def run_case(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    retrieval=None,
    sources=(),
    forbid_sources=(),
    paths=(),
    forbid_paths=(),
    disallow=(),
    answer_terms=(),
    generated_function=None,
    generated_parameters=None,
    generated_calls=(),
    definitions=(),
    expected_outcome="completed",
    choice="default",
    confirm="quick",
):
    """Run and record a full RAG turn with explicit context expectations.

    Parameters
    ----------
    context : dict
        Test conversation returned by ``conversation``.
    case_id : str
        Unique identifier used for later comparisons and follow-ups.
    question : str
        Unmodified user question.
    intent : str or None, optional
        Expected code intent; None records an exploratory observation.
    standalone : bool or None, optional
        Expected standalone classification; None omits this check.
    related_to : tuple of str, optional
        Earlier case IDs whose turns must be linked by the router.
    forbid_related : tuple of str, optional
        Earlier case IDs the router should not link to this question.
    retrieval : str or None, optional
        Expected retrieval mode: search, reuse, document, or none.
    sources : tuple of str, optional
        Source filenames that must appear in saved evidence.
    forbid_sources : tuple of str, optional
        Source filenames that must not occur in saved evidence.
    paths : tuple of str, optional
        Exact source-relative file paths expected in saved evidence.
    forbid_paths : tuple of str, optional
        Paths that must not be retrieved.
    disallow : tuple of str, optional
        Intents that must not be chosen, including ambiguous cases.
    answer_terms : tuple of str, optional
        Required final-answer keywords. A lightweight sanity check, not
        a semantic evaluation of code correctness.
    generated_function : str or None, optional
        Expected Python function name (compiled, never executed).
    generated_parameters : tuple of str or None, optional
        Expected synchronous positional parameter names, with no extras.
    generated_calls : tuple of str, optional
        Expected API-call expressions in that function's AST.
    definitions : tuple of tuple of str, optional
        Source-relative file and qualified symbol pairs. Missing references
        produce SKIPPED cases; present definition lines must be retrieved.
    expected_outcome : str, optional
        Completed by default; use cancelled for a deliberate cancel test.
    choice : str or tuple, optional
        Scripted GUI document/list selection.
    confirm : str, optional
        GUI-equivalent confirmation mode, with network access disabled.

    Returns
    -------
    dict
        Recorded answer and machine-verifiable routing observations.

    Notes
    -----
    A routing mismatch is recorded without stopping the conversation.
    Later turns therefore test how the system handles imperfect history.
    Missing required previous turns is a test setup error and raises.

    """
    if intent is not None and intent not in code_intents():
        raise ValueError(f"Unknown intent: {intent}")
    expected_links = _expected_links(context, related_to)
    forbidden_links = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    facts = [_definition(*item) for item in definitions]
    missing = [item for item, fact in zip(definitions, facts) if fact is None]
    if missing:
        return skip_case(
            context["label"], case_id, question, intent=intent,
            model=context["model"], router=context["router"],
            reason=f"Stale source-reference prerequisites: {missing}",
        )
    session = context["session"]
    before = len(session.turns)
    previous = _previous_snapshot(context)
    events = []
    choose, confirm_plan = _gui_callbacks(events, choice, confirm)
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "full",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": expected_links,
        "forbidden_related_ids": forbidden_links,
        "expected_retrieval": retrieval,
        "expected_sources": list(sources),
        "forbidden_sources": list(forbid_sources),
        "expected_paths": list(paths),
        "forbidden_paths": list(forbid_paths),
        "disallowed_intents": list(disallow),
        "expected_answer_terms": list(answer_terms),
        "expected_generated_function": generated_function,
        "expected_generated_parameters": (
            list(generated_parameters)
            if generated_parameters is not None else None
        ),
        "expected_generated_calls": list(generated_calls),
        "expected_definitions": facts,
        "expected_outcome": expected_outcome,
        "confirmation_policy": confirm,
        "effective_settings": _effective_settings(session),
        "previous": previous,
        "trace_path": str(context["path"].relative_to(project_dir)),
        "gui_events": events,
    }
    try:
        answer = session.ask(
            question,
            retrieval_count=RETRIEVAL_COUNT,
            generation_settings=dict(GENERATION_SETTINGS),
            choice_callback=choose,
            confirm_callback=confirm_plan,
        )
        if len(session.turns) != before + 1:
            raise RuntimeError("Expected exactly one committed turn.")
        turn = session.turns[-1]
        parameters = turn.parameters
        tool = parameters.get("tool") or {}
        native = _native_route(context["path"], len(session.turns))
        representation = _recorded_representation(
            turn, native.get("representation")
        )
        internal = tool.get("internal_generation") or {}
        attempts = internal.get("calls")
        record.update(
            {
                "answer": answer,
                "answer_summary": representation["summary"],
                "summary_source": representation["origin"],
                "summary_consistent": representation["consistent"],
                "summary_sources": representation["sources"],
                "turn_number": len(session.turns),
                "intent": parameters.get("intent"),
                "tool": tool.get("name"),
                "tool_status": tool.get("status"),
                "tool_summary": tool,
                "standalone": parameters.get("standalone"),
                "related_turn_ids": list(
                    parameters.get("related_turn_ids", [])
                ),
                "history_turn_ids": list(
                    parameters.get("history_turn_ids", [])
                ),
                "retrieval_mode": parameters.get("retrieval_mode"),
                "sources": _source_names(turn.evidence),
                "source_paths": _source_paths(turn.evidence),
                "evidence_count": len(turn.evidence),
                "evidence_spans": [
                    {"path": str(item.path),
                     "start_line": item.start_line,
                     "end_line": item.end_line}
                    for item in turn.evidence
                ],
                "files_scanned": tool.get("files_scanned"),
                "files_with_mentions": tool.get(
                    "files_with_mentions"
                ),
                "listed": tool.get("listed"),
                "passages_read": tool.get("passages_read"),
                "planned_passages": tool.get("passages"),
                "generation_events": (
                    len(native.get("model_calls", []))
                    if native.get("trace_available") else None
                ),
                "tool_internal_attempts": (
                    len(attempts) if isinstance(attempts, list)
                    else None
                ),
                "invalid_citations": parameters.get("invalid_citations", []),
                "completion": parameters.get("completion"),
                "usage": parameters.get("usage", {}),
                "finish_reason": parameters.get("finish_reason"),
                "trace": native,
            }
        )
        _grade_record(record)
        context["case_turn_ids"][case_id] = len(session.turns)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "outcome": "failed", "operation_selected": False,
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record


def probe_route(
    context,
    case_id,
    question,
    *,
    intent=None,
    standalone=None,
    related_to=(),
    forbid_related=(),
    disallow=(),
):
    """Test routing against real conversation history without execution.

    Parameters
    ----------
    context : dict
        Active context with previously saved conversation turns.
    case_id : str
        Unique diagnostic case ID.
    question : str
        Question to route without saving or answering.
    intent : str or None, optional
        Intended router classification.
    standalone : bool or None, optional
        Expected independence from previous turns.
    related_to : tuple of str, optional
        Required previous case IDs.
    forbid_related : tuple of str, optional
        Previous case IDs that must not be linked.
    disallow : tuple of str, optional
        Intents that must not be selected for this request.

    Returns
    -------
    dict
        Routing-only decision and any diagnosed mismatches.

    Notes
    -----
    This uses KíLín's own ``route_question`` and the active session's
    private router accessor. It does not save an answer or execute a tool.

    """
    if intent is not None and intent not in code_intents():
        raise ValueError(f"Unknown intent: {intent}")
    required = _expected_links(context, related_to)
    forbidden = _expected_links(context, forbid_related)
    if case_id in CASE_INDEX:
        raise ValueError(f"Repeated case ID: {case_id}")

    session = context["session"]
    started = perf_counter()
    record = {
        "id": case_id,
        "mode": "route-only",
        "label": context["label"],
        "model": session.chat_session.model_id,
        "router_model": session.router_model,
        "question": question,
        "expected_intent": intent,
        "expected_tool": expected_tool(intent),
        "expected_standalone": standalone,
        "expected_related_ids": required,
        "forbidden_related_ids": forbidden,
        "disallowed_intents": list(disallow),
        "previous": _previous_snapshot(context),
        "effective_settings": _effective_settings(session),
        "trace_path": str(context["path"].relative_to(project_dir)),
    }
    try:
        config = project_type(session.project.profile.type_id)
        mark = session.usage_mark()
        decision = route_question(
            session._router_chat(), config, session.turns, question
        )
        record.update(
            {
                "intent": decision.intent,
                "tool": None,
                "standalone": decision.standalone,
                "related_turn_ids": list(decision.related_turn_ids),
                "retrieval_mode": config.intents[
                    decision.intent
                ].retrieval_mode,
                "sources": [],
                "answer": "(routing only: no answer)",
                "usage": session.usage_since(mark),
                "trace": {
                    "shortcut": decision.shortcut,
                    "fallback": decision.fallback,
                    "router_raw": decision.raw,
                    "dropped_turns": list(decision.dropped_turns),
                    "refers_back": decision.refers_back,
                },
            }
        )
        _grade_record(record)
    except Exception as error:
        record.update(
            {
                "answer": "",
                "intent": None,
                "sources": [],
                "status": "ERROR",
                "outcome": "failed", "operation_selected": False,
                "issues": [f"{type(error).__name__}: {error}"],
            }
        )
    record["seconds"] = round(perf_counter() - started, 3)
    _register(record)
    return record


def skip_case(
    label, case_id, question, *, reason, mode="full", intent=None,
    model=None, router=None,
):
    """Export an explicit skipped case without claiming model execution.

    Parameters
    ----------
    label : str
        Scenario group, including failed or unavailable model groups.
    case_id : str
        Unique planned case identifier.
    question : str
        Planned prompt, retained for comparing denominators.
    reason : str
        Model setup failure, missing source reference or invalid antecedent.
    mode : str, optional
        Intended full or routing-only execution mode.
    intent : str or None, optional
        Intended routing class.
    model : str or None, optional
        Answering model key, defaulting to PRIMARY_MODEL.
    router : str or None, optional
        Requested router identifier.

    Returns
    -------
    dict
        Saved SKIPPED record; no trace turn, answer or model usage is claimed.

    """
    record = {
        "id": case_id, "mode": mode, "label": label,
        "model": MODEL_IDS[model or PRIMARY_MODEL],
        "router_model": router or ROUTER_MODEL,
        "question": question, "expected_intent": intent,
        "expected_tool": expected_tool(intent), "intent": None,
        "tool": None, "status": "SKIPPED", "outcome": "skipped",
        "scope_status": "---", "operation_selected": False,
        "skip_reason": reason, "answer": "", "sources": [],
        "seconds": 0.0, "issues": [f"skipped: {reason}"],
        "previous": [], "trace": {}, "checks": {},
    }
    _register(record)
    return record


def _seed_is_valid(record):
    """Check objective antecedent prerequisites, not semantic truth.

    Parameters
    ----------
    record : dict
        Completed seed record with scope and expected-operation checks.

    Returns
    -------
    bool
        True for the expected completed task with required evidence and a
        nonempty final response. Wrong routes cannot seed comparisons.

    """
    return (
        record.get("status") == "MATCH"
        and record.get("operation_selected") is True
        and record.get("outcome") == "completed"
        and not record.get("completion")
        and not any(record.get("checks", {}).get(kind)
                    for kind in ("scope", "content", "trace"))
        and bool(strip_thinking(record.get("answer") or "").strip())
    )


In [6]:
def show_results(*, prefix=None, group=None, mode=None):
    """Display a compact summary across selected recorded cases.

    Parameters
    ----------
    prefix : str or None, optional
        Limit case IDs to a prefix.
    group : str or None, optional
        Limit records to a named conversation group.
    mode : str or None, optional
        Limit to ``full`` or ``route-only`` cases.

    Returns
    -------
    list of dict
        Selected records.

    """
    rows = [
        record
        for record in CASE_RESULTS
        if (prefix is None or record["id"].startswith(prefix))
        and (group is None or record["label"] == group)
        and (mode is None or record["mode"] == mode)
    ]
    if not rows:
        print("No matching cases recorded.")
        return []
    headings = (
        "Case",
        "Model",
        "Expected",
        "Actual",
        "Links",
        "Route",
        "Execution",
        "Scope",
        "Evidence files",
        "Warnings",
        "Seconds",
    )
    header = "".join(f"<th>{html.escape(name)}</th>" for name in headings)
    body = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 65)}</td>"
            for value in (
                item["id"],
                item.get("model", "-"),
                item.get("expected_intent") or "---",
                item.get("intent") or "-",
                str(item.get("related_turn_ids", [])),
                item["status"],
                item.get("outcome", "---"),
                item.get("scope_status", "---"),
                (
                    f"{len(item.get('sources', []))} retained / "
                    f"{item['files_scanned']} scanned"
                    if item.get("files_scanned") is not None
                    else str(len(item.get("sources", [])))
                    if item["mode"] == "full" else "---"
                ),
                ", ".join(
                    key for key in ("execution", "trace", "content")
                    if item.get("checks", {}).get(key)
                ) or "---",
                f"{item['seconds']:.2f}",
            )
        )
        + "</tr>"
        for item in rows
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:12px'>"
            "<table style='border-collapse:collapse;width:100%'>"
            f"<thead><tr>{header}</tr></thead><tbody>{body}</tbody>"
            "</table></div>"
        )
    )
    return rows


def show_coverage():
    """Keep attempted, selected and completed coverage separate.

    Returns
    -------
    dict
        Counts per configured intent. Completion requires selection of the
        expected operation; cancellation never counts as task completion.
        Outcomes describe execution, not scientific correctness.

    """
    coverage = {}
    for intent, settings in code_intents().items():
        records = [item for item in CASE_RESULTS
                   if item.get("expected_intent") == intent]
        full = [item for item in records if item["mode"] == "full"]
        selected = [item for item in full if item.get("operation_selected")]
        coverage[intent] = {
            "tool": (settings.get("tool") or {}).get("name"),
            "attempted": sum(item.get("outcome") != "skipped"
                             for item in full),
            "probes": sum(item["mode"] == "route-only"
                          and item.get("outcome") != "skipped"
                          for item in records),
            "selected": len(selected),
            "completed": sum(item.get("outcome") == "completed"
                             for item in selected),
            "partial": sum(item.get("outcome") == "partial" for item in full),
            "cancelled": sum(item.get("outcome") == "cancelled"
                             for item in full),
            "failed": sum(item.get("outcome") in ("failed", "incomplete")
                          for item in full),
            "skipped": sum(item.get("outcome") == "skipped"
                           for item in records),
        }
    headers = ("Intent", "Tool", "Full attempts", "Probes", "Selected",
               "Completed", "Partial", "Cancelled", "Failed", "Skipped")
    rows = "".join(
        "<tr>" + "".join(f"<td>{_brief(value)}</td>" for value in (
            intent, item["tool"] or "---", item["attempted"], item["probes"],
            item["selected"], item["completed"], item["partial"],
            item["cancelled"], item["failed"], item["skipped"],
        )) + "</tr>" for intent, item in coverage.items()
    )
    head = "".join(f"<th>{html.escape(name)}</th>" for name in headers)
    display(HTML(
        "<div style='overflow-x:auto;font-size:12px'><table>"
        f"<thead><tr>{head}</tr></thead><tbody>{rows}</tbody></table></div>"
    ))
    return coverage


def show_answer_model_comparison():
    """Compare the same numbered prompts across answering models.

    Returns
    -------
    list of dict
        Matching full-pipeline answering-model records.

    """
    items = [item for item in CASE_RESULTS if item["id"].startswith("M_")]
    if not items:
        print("No answering-model comparison has been run.")
        return []
    items = sorted(
        items, key=lambda item: (item["id"].rsplit("_", 1)[-1], item["model"])
    )
    headers = (
        "Prompt",
        "Answer model",
        "Intent",
        "Sources",
        "Seconds",
        "Recorded summary",
        "Route",
        "Execution",
    )
    head = "".join(f"<th>{html.escape(name)}</th>" for name in headers)
    rows = "".join(
        "<tr>"
        + "".join(
            f"<td>{_brief(value, 145)}</td>"
            for value in (
                item["id"].rsplit("_", 1)[-1],
                item["model"],
                item.get("intent"),
                ", ".join(item.get("sources", [])),
                f"{item['seconds']:.2f}",
                item.get("answer_summary") or "(summary unavailable)",
                item["status"],
                item.get("outcome", "---"),
            )
        )
        + "</tr>"
        for item in items
    )
    display(
        HTML(
            "<div style='overflow-x:auto;font-size:11px'>"
            "<table style='width:100%;border-collapse:collapse'>"
            f"<thead><tr>{head}</tr></thead><tbody>{rows}</tbody>"
            "</table></div>"
        )
    )
    return items


def show_diagnostic_totals():
    """Print routing outcomes and separate evidence/trace warnings.

    Returns
    -------
    dict
        Counts by execution mode and diagnostic category.

    """
    totals = {}
    for mode in ("full", "route-only"):
        items = [item for item in CASE_RESULTS if item["mode"] == mode]
        totals[mode] = {
            "cases": len(items),
            "match": sum(item["status"] == "MATCH" for item in items),
            "review": sum(item["status"] == "REVIEW" for item in items),
            "unscored": sum(item["status"] == "---" for item in items),
            "errors": sum(item["status"] == "ERROR" for item in items),
            "skipped": sum(item["status"] == "SKIPPED" for item in items),
            "scope": sum(bool(item.get("checks", {}).get("scope"))
                         for item in items),
            "citation_warnings": sum(
                bool(item.get("invalid_citations")) for item in items
            ),
            "trace_warnings": sum(
                bool(item.get("checks", {}).get("trace"))
                for item in items
            ),
            "content_anchors": sum(
                bool(item.get("checks", {}).get("content"))
                for item in items
            ),
        }
    display(HTML(
        "<table style='font-size:12px;border-collapse:collapse'>"
        "<tr>" + "".join(f"<th>{name}</th>" for name in (
            "Mode", "Cases", "Match", "Review", "---", "Errors", "Skipped",
            "Scope flags", "Citation flags", "Trace flags",
            "Anchor flags"
        )) + "</tr>" + "".join(
            "<tr>" + "".join(f"<td>{_brief(value)}</td>" for value in (
                mode, item["cases"], item["match"], item["review"],
                item["unscored"], item["errors"], item["skipped"],
                item["scope"],
                item["citation_warnings"], item["trace_warnings"],
                item["content_anchors"]
            )) + "</tr>" for mode, item in totals.items()
        ) + "</table>"
    ))
    return totals


def verify_saved_workspace():
    """Compare notebook full-turn records with committed trace checkpoints.

    Returns
    -------
    dict
        Number of verified answers and any field disagreements.

    Notes
    -----
    Routing probes have no saved turn and are deliberately excluded.

    """
    checked = 0
    problems = []
    cache = {}
    for record in CASE_RESULTS:
        if record["mode"] != "full" or record["status"] in (
            "ERROR", "SKIPPED"
        ):
            continue
        path = project_dir / record["trace_path"]
        if path not in cache:
            try:
                with path.open(encoding="utf-8") as stream:
                    trace = json.load(stream)
                cache[path] = trace["sessions"]["05_conversations"][
                    "resume_state"
                ]["turns"]
            except (OSError, ValueError, KeyError, TypeError):
                cache[path] = []
        number = record.get("turn_number", 0)
        saved = (
            cache[path][number - 1]
            if 1 <= number <= len(cache[path]) else None
        )
        if saved is None:
            errors = ["missing committed turn"]
        else:
            params = saved.get("parameters", {})
            errors = []
            for field, observed in (
                ("answer", record.get("answer")),
                ("question", record.get("question")),
            ):
                if saved.get(field) != observed:
                    errors.append(field)
            for field in ("intent", "related_turn_ids", "history_turn_ids"):
                if params.get(field) != record.get(field):
                    errors.append(field)
            if len(saved.get("evidence", [])) != record.get(
                "evidence_count"
            ):
                errors.append("evidence count")
            if _source_paths(saved.get("evidence", [])) != (
                record.get("source_paths")
            ):
                errors.append("source paths")
            if _source_names(saved.get("evidence", [])) != record.get(
                "sources"
            ):
                errors.append("evidence sources")
            if (params.get("tool") or {}).get("name") != record.get(
                "tool"
            ):
                errors.append("tool")
            if params.get("retrieval_mode") != record.get(
                "retrieval_mode"
            ):
                errors.append("retrieval mode")
            if (params.get("representation") or {}).get(
                "summary"
            ) != record.get("answer_summary"):
                errors.append("recorded summary")
        record["workspace_mismatches"] = errors
        checked += 1
        if errors:
            problems.append({"case": record["id"], "fields": errors})
    print("Committed turns verified:", checked)
    print("Notebook/workspace discrepancies:", len(problems))
    for item in problems:
        print(f"  {item['case']}: {', '.join(item['fields'])}")
    return {"checked": checked, "mismatches": problems}


def annotate_case(case_id, *, quality=None, note=""):
    """Record a manual code-quality verdict without model grading.

    Parameters
    ----------
    case_id : str
        Existing case ID.
    quality : str or None, optional
        Human judgment such as ``correct``, ``mixed``, or ``incorrect``.
    note : str, optional
        Source lines or tests supporting the assessment.

    """
    CASE_INDEX[case_id]["manual_review"] = {
        "quality": quality,
        "note": note,
    }


CODE_REFERENCE_CASES = (
    ("D02", "kilin/rag.py", "RAGSession.ask"),
    ("D03", "kilin/conversations.py", "ConversationStore.rename"),
    ("A01", "kilin/answering.py", "route_question"),
    ("A03", "kilin/workspace.py", "prepare_sources_workspace"),
    ("H01", "kilin/gui/web/app.js", None),
    ("H02", "kilin/gui/web/index.html", None),
    ("H03", "kilin/project_types/code.json", None),
    ("H04", "tutorials/150_logging_trace.ipynb", None),
    ("X01", "kilin/rag.py", "RAGSession.ask"),
    ("X02", "kilin/conversations.py", "ConversationStore.create"),
)


def show_code_references():
    """Display independent definitions and per-case evidence coverage.

    Returns
    -------
    list of dict
        Reference-case records that have already executed.

    Notes
    -----
    AST definitions are computed from this experiment's copied files.
    Found words and source passages are insufficient to prove an
    explanation of control flow, side effects, or cross-file behavior.

    """
    rows = []
    observations = []
    for case_id, path, symbol in CODE_REFERENCE_CASES:
        item = CASE_INDEX.get(case_id)
        fact = (AST_FACTS.get((path, symbol)) if symbol else None)
        if item is not None:
            observations.append(item)
        source_ok = (
            "yes" if item and _has_source(
                item.get("source_paths", []), path
            ) else "no" if item else "not run"
        )
        ground = (
            f"{fact['kind']} at line {fact['line']}" if fact
            else "whole file" if symbol is None else "not in snapshot"
        )
        quality = (
            (item.get("manual_review") or {}).get("quality")
            if item else None
        ) or "pending"
        values = (case_id, path, symbol or "(file)", ground,
                  source_ok, quality)
        rows.append(
            "<tr>" + "".join(
                f"<td>{_brief(value, 105)}</td>" for value in values
            ) + "</tr>"
        )
    headers = (
        "Case", "Reference file", "Symbol", "Snapshot fact",
        "In evidence", "Manual quality",
    )
    head = "".join(f"<th>{html.escape(h)}</th>" for h in headers)
    display(HTML(
        "<div style='overflow-x:auto;font-size:12px'>"
        "<table style='border-collapse:collapse;width:100%'>"
        f"<thead><tr>{head}</tr></thead>"
        f"<tbody>{''.join(rows)}</tbody></table></div>"
    ))
    return observations


def export_evaluation():
    """Write local JSON/CSV records and the experiment manifest.

    Returns
    -------
    pathlib.Path
        Directory holding the saved evaluation files.

    """
    folder = project_dir / "outputs" / "notebook_evaluation"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "cases.json").write_text(
        json.dumps(CASE_RESULTS, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    with (folder / "cases.csv").open(
        "w", encoding="utf-8", newline=""
    ) as stream:
        columns = [
            "id",
            "mode",
            "label",
            "model",
            "router_model",
            "expected_intent",
            "intent",
            "expected_tool",
            "tool",
            "standalone",
            "related_turn_ids",
            "history_turn_ids",
            "retrieval_mode",
            "sources",
            "evidence_count",
            "source_paths",
            "expected_paths",
            "forbidden_paths",
            "expected_generated_function",
            "files_scanned",
            "files_with_mentions",
            "passages_read",
            "generation_events",
            "tool_internal_attempts",
            "status",
            "outcome",
            "operation_selected",
            "expected_outcome",
            "skip_reason",
            "effective_settings",
            "generated_check",
            "evidence_spans",
            "issues",
            "seconds",
            "trace_path",
            "answer_summary",
            "summary_source",
            "summary_consistent",
            "scope_status",
            "checks",
            "invalid_citations",
            "completion",
            "workspace_mismatches",
            "manual_review",
        ]
        assert len(columns) == len(set(columns))
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for record in CASE_RESULTS:
            writer.writerow(
                {
                    name: (
                        json.dumps(record.get(name), ensure_ascii=False)
                        if isinstance(record.get(name), (list, dict))
                        else record.get(name)
                    )
                    for name in columns
                }
            )
    manifest = {
        "experiment_id": EXPERIMENT_ID,
        "project": str(project_dir),
        "primary_model": MODEL_IDS[PRIMARY_MODEL],
        "router_model": ROUTER_MODEL,
        "comparison_models": [MODEL_IDS[k] for k in COMPARISON_MODELS],
        "comparison_routers": list(COMPARISON_ROUTERS),
        "backend": BACKEND,
        "requested_generation": dict(GENERATION_SETTINGS),
        "embedding_model": project.profile.embedding_model(BACKEND),
        "profile_sha256": sha256(
            (project_dir / "profile.ini").read_bytes()
        ).hexdigest(),
        "code_graph": project_type("code").code_graph,
        "startup_records": STARTUP_RECORDS,
        "infrastructure_checks": INFRASTRUCTURE_RESULTS,
        "allow_download": ALLOW_DOWNLOAD,
        "retrieval_count": RETRIEVAL_COUNT,
        "source_hashes": globals().get("SOURCE_HASHES", {}),
        "source_version": SOURCE_VERSION,
        "source_directories": ["kilin", "tutorials"],
        "extensions": list(CODE_EXTENSIONS),
        "code_intents": list(code_intents()),
        "summary_source": "conversation.turn.representation",
        "enabled_sections": {
            "long_tools": RUN_LONG_TOOLS,
            "file_review": RUN_FULL_SOURCE_READ,
            "code_review": RUN_CODE_REVIEW,
            "answer_models": RUN_MODEL_COMPARISON,
            "router_models": RUN_ROUTER_COMPARISON,
            "persistence": RUN_PERSISTENCE_CHECK,
        },
        "conversations": CONVERSATION_INDEX,
        "ast_facts": list(globals().get("AST_FACTS", {}).values()),
        "ast_relations": globals().get("AST_RELATIONS", {}),
    }
    (folder / "manifest.json").write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    print("Exported:", folder)
    return folder


### 2.1 Protocol preflight

The code project currently defines **14 intents**. If that vocabulary changes
in a development checkout, update the notebook's coverage contract rather
than silently treating an untested intent as covered. Tool names come from
the active project configuration, not hard-coded copies.


In [7]:
PLANNED_INTENTS = {
    "small_talk", "code_explanation", "fact_finding", "code_generation",
    "debugging", "file_reading", "project_inventory", "mention_listing",
    "topic_listing", "mention_classification", "refine_results",
    "source_context", "conversation_followup", "rephrase_previous",
}
configured = set(code_intents())
assert PLANNED_INTENTS == configured, (
    "Code intent vocabulary changed. Revisit coverage. "
    f"Missing: {sorted(PLANNED_INTENTS - configured)}; "
    f"new: {sorted(configured - PLANNED_INTENTS)}"
)
print("Configured code intents:", len(configured))
show_coverage()


Configured code intents: 14


Intent,Tool,Full attempts,Probes,Selected,Completed,Partial,Cancelled,Failed,Skipped
code_explanation,---,0,0,0,0,0,0,0,0
fact_finding,find_fact,0,0,0,0,0,0,0,0
code_generation,---,0,0,0,0,0,0,0,0
debugging,---,0,0,0,0,0,0,0,0
file_reading,batch_review,0,0,0,0,0,0,0,0
project_inventory,project_inventory,0,0,0,0,0,0,0,0
mention_listing,scan_and_list,0,0,0,0,0,0,0,0
topic_listing,find_passages,0,0,0,0,0,0,0,0
mention_classification,batch_classify,0,0,0,0,0,0,0,0
refine_results,filter_results,0,0,0,0,0,0,0,0


{'code_explanation': {'tool': None,
  'attempted': 0,
  'probes': 0,
  'selected': 0,
  'completed': 0,
  'partial': 0,
  'cancelled': 0,
  'failed': 0,
  'skipped': 0},
 'fact_finding': {'tool': 'find_fact',
  'attempted': 0,
  'probes': 0,
  'selected': 0,
  'completed': 0,
  'partial': 0,
  'cancelled': 0,
  'failed': 0,
  'skipped': 0},
 'code_generation': {'tool': None,
  'attempted': 0,
  'probes': 0,
  'selected': 0,
  'completed': 0,
  'partial': 0,
  'cancelled': 0,
  'failed': 0,
  'skipped': 0},
 'debugging': {'tool': None,
  'attempted': 0,
  'probes': 0,
  'selected': 0,
  'completed': 0,
  'partial': 0,
  'cancelled': 0,
  'failed': 0,
  'skipped': 0},
 'file_reading': {'tool': 'batch_review',
  'attempted': 0,
  'probes': 0,
  'selected': 0,
  'completed': 0,
  'partial': 0,
  'cancelled': 0,
  'failed': 0,
  'skipped': 0},
 'project_inventory': {'tool': 'project_inventory',
  'attempted': 0,
  'probes': 0,
  'selected': 0,
  'completed': 0,
  'partial': 0,
  'cancelled'

## 3. Create an initially empty code project

An empty project provides an essential negative control: it can answer
greetings and report that no source files have been imported, but cannot
legitimately cite KíLín code. Only the **experiment profile** gains the
seven selected extensions. KíLín's packaged type definition is unchanged.


In [8]:
from kilin.workspace import prepare_sources_workspace

if project_dir.exists():
    raise FileExistsError(
        f"Experiment workspace already exists: {project_dir}"
    )
project = prepare_sources_workspace(
    [], "code", project_dir,
    settings={
        "extensions": list(CODE_EXTENSIONS),
        "backend": BACKEND,
        "chat_mlx": MODEL_IDS[PRIMARY_MODEL],
        **GENERATION_SETTINGS,
        "allow_download": ALLOW_DOWNLOAD,
        "retrieval_count": RETRIEVAL_COUNT,
    },
)
assert set(project.profile.extensions) == set(CODE_EXTENSIONS)
assert list(project.source.iterdir()) == []
print("Created empty code project:", project.root)
print("Saved code extensions:", project.profile.extensions)

# Default comparisons and reports use the actual saved choice.
ROUTER_MODEL = kilin.read_router_model(project_dir, "mlx")
print("Effective router:", ROUTER_MODEL)


Created empty code project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_code_routing_20261109-161523
Saved code extensions: ('.py', '.ipynb', '.m', '.md', '.json', '.js', '.html')
Effective router: Qwen/Qwen3-4B-MLX-4bit


### 3.1 Empty-project behavior

A source-free answer is useful for detecting unsupported citations and
fabricated code details. These controls are deliberately run before any
project files are imported.


In [9]:
with conversation("empty-project") as ctx:
    run_case(
        ctx, "E01", "hello", intent="small_talk",
        standalone=True, retrieval="none",
    )
    run_case(
        ctx, "E02", "Which source files are in this project?",
        intent="project_inventory", retrieval="none",
    )
    run_case(
        ctx, "E03", "Explain RAGSession.ask as implemented here.",
        intent="code_explanation", retrieval="search",
    )
    run_case(
        ctx, "E04", "Find all mentions of ConversationStore in this "
        "empty project.", intent="mention_listing",
    )
show_results(prefix="E")


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Conversation empty-project: conversation_2026-11-09T15-15-23.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
small_talk → small_talk,---,need []; got [],none,0:,187/3,1.93s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
project_inventory → project_inventory,project_inventory,need []; got [],none,0:,1153/14,3.24s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,0:,1525/438,18.95s,MATCH,completed,execution


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [],none,0: (0 scanned),1498/30,4.07s,MATCH,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
E01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,completed,---,0,---,1.93
E02,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,completed,---,0,---,3.24
E03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,---,0,execution,18.95
E04,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,---,0 retained / 0 scanned,---,4.07


[{'id': 'E01',
  'mode': 'full',
  'label': 'empty-project',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'hello',
  'expected_intent': 'small_talk',
  'expected_tool': None,
  'expected_standalone': True,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': 'none',
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': [],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ',
   'generation': {'temperature': 0.0,
    'thinking': True,
    'max_new

## 4. Import the development snapshot and establish ground truth

Import only the `kilin/` package and the `tutorials/` notebooks into the
managed workspace. The normal scanner applies the saved extensions and
file-selection limits. The notebook records the hashes of **copied** files
and verifies them against the development originals, so later changes to
the checkout cannot silently alter this experiment.

AST reference facts are derived from those copied files, not from KíLín's
answers. They identify actual definitions and their line numbers; they do
not establish runtime behavior or semantic correctness. JavaScript, JSON,
and HTML are ordinary source files in this experiment.

Selection is checked before copying using the normal source-selection API,
then each copied file is hashed against the original. Independent extraction
reports files with no content units. Definition and relationship references
come from the copied Python AST; missing references are marked stale rather
than guessed. These checks do not import the analyzed source modules.

In [10]:
import ast
from hashlib import sha256

from kilin.workspace import add_project_sources

sources = [repo_root / "kilin", repo_root / "tutorials"]
if not all(path.is_dir() for path in sources):
    raise FileNotFoundError("Missing kilin/ or tutorials/ source folder.")
from kilin.repository import scan_repository
from kilin.sources import read_repository_files, extract_source_units

EXPECTED_SOURCE_FILES = set()
for folder in sources:
    scan = scan_repository(
        folder, extensions=project.profile.extensions,
        max_file_bytes=project.profile.max_file_bytes,
    )
    if not scan["complete"]:
        raise RuntimeError(f"Incomplete source inventory: {folder}")
    EXPECTED_SOURCE_FILES.update(
        f"{folder.name}/{path}" for path in scan["selected_files"]
    )
project = add_project_sources(project_dir, sources)

SOURCE_HASHES = {}
for subdir in ("kilin", "tutorials"):
    snapshot = project.source / subdir
    assert snapshot.is_dir(), f"Missing imported directory: {subdir}"
    for copied in sorted(snapshot.rglob("*")):
        if not copied.is_file():
            continue
        relative = copied.relative_to(project.source)
        original = repo_root / relative
        digest = sha256(copied.read_bytes()).hexdigest()
        SOURCE_HASHES[relative.as_posix()] = digest
        if not original.is_file():
            raise FileNotFoundError(f"Source disappeared: {original}")
        if sha256(original.read_bytes()).hexdigest() != digest:
            raise RuntimeError(f"Source changed during copy: {relative}")

assert all(
    name.startswith(("kilin/", "tutorials/"))
    for name in SOURCE_HASHES
)
assert (project.source / "kilin/gui/web/app.js").is_file()
assert (project.source / "kilin/gui/web/index.html").is_file()
assert (project.source / "kilin/project_types/code.json").is_file()
assert (project.source / "tutorials/150_logging_trace.ipynb").is_file()
for extension in (".json", ".js", ".html"):
    assert any(name.endswith(extension) for name in SOURCE_HASHES)

print("Imported source files:", len(SOURCE_HASHES))
print("Python modules:", sum(n.endswith(".py") for n in SOURCE_HASHES))
print("Tutorials:", sum(n.startswith("tutorials/") for n in SOURCE_HASHES))
print("JavaScript/JSON/HTML:", sum(
    n.endswith((".js", ".json", ".html"))
    for n in SOURCE_HASHES
))
print("Source hashes recorded; originals remain unchanged.")

assert set(SOURCE_HASHES) == EXPECTED_SOURCE_FILES
source_scan = scan_repository(
    project.source, extensions=project.profile.extensions,
    max_file_bytes=project.profile.max_file_bytes,
)
assert source_scan["complete"]
assert set(source_scan["selected_files"]) == EXPECTED_SOURCE_FILES
source_files = read_repository_files(source_scan)
source_units = extract_source_units(source_files)
INDEXABLE_PATHS = sorted({str(unit.path) for unit in source_units})
INFRASTRUCTURE_RESULTS["ingestion"] = {
    "selected_files": len(EXPECTED_SOURCE_FILES),
    "copied_files": len(SOURCE_HASHES),
    "read_files": len(source_files), "units": len(source_units),
    "files_without_units": sorted(EXPECTED_SOURCE_FILES
                                 - set(INDEXABLE_PATHS)),
}
print("Files with source units:", len(INDEXABLE_PATHS))
print("Files without content units:",
      INFRASTRUCTURE_RESULTS["ingestion"]["files_without_units"])
del source_files, source_units


Imported source files: 205
Python modules: 143
Tutorials: 23
JavaScript/JSON/HTML: 20
Source hashes recorded; originals remain unchanged.
Files with source units: 205
Files without content units: []


In [11]:
def _definition(path, dotted_name):
    """Find a definition without importing or executing the target module.

    Parameters
    ----------
    path : str
        Source-relative Python file in the copied workspace.
    dotted_name : str
        Top-level symbol or class method.

    Returns
    -------
    dict or None
        Current source identity and line span, or None for a stale reference.

    """
    file = project.source / path
    if not file.is_file():
        return None
    try:
        tree = ast.parse(file.read_text(encoding="utf-8"))
    except (SyntaxError, UnicodeError):
        return None
    body = tree.body
    for part in dotted_name.split("."):
        node = next((item for item in body if isinstance(
            item, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)
        ) and item.name == part), None)
        if node is None:
            return None
        body = node.body
    return {
        "file": path, "symbol": dotted_name,
        "kind": type(node).__name__, "line": node.lineno,
        "end_line": node.end_lineno,
    }


REFERENCE_SYMBOLS = (
    ("kilin/rag.py", "RAGSession.ask"),
    ("kilin/rag.py", "RAGSession"),
    ("kilin/conversations.py", "ConversationStore.create"),
    ("kilin/conversations.py", "ConversationStore.rename"),
    ("kilin/answering.py", "route_question"),
    ("kilin/answering.py", "select_history"),
    ("kilin/workspace.py", "prepare_sources_workspace"),
    ("kilin/project.py", "configure_project"),
    ("kilin/project_type.py", "project_type"),
)
AST_FACTS = {pair: _definition(*pair) for pair in REFERENCE_SYMBOLS}
for pair, fact in AST_FACTS.items():
    print(f"{fact['file']}:{fact['line']} {fact['symbol']}" if fact
          else f"STALE reference: {pair[0]}:{pair[1]}")

# Derive selected simple relationships directly from Python syntax, not
# from Kilin's graph or answers. Unavailable expectations remain explicit.
AST_RELATIONS = []
for path, symbol, relation, target in (
    ("kilin/rag.py", None, "imports", "kilin.answering.select_history"),
    ("kilin/rag.py", "RAGSession.ask", "calls", "select_history"),
    ("kilin/_model_completion.py", "IncompleteModelResponse",
     "inherits", "RuntimeError"),
):
    file = project.source / path
    node = ast.parse(file.read_text()) if file.is_file() else None
    if symbol and node:
        for name in symbol.split("."):
            node = next((item for item in getattr(node, "body", [])
                         if getattr(item, "name", None) == name), None)
    observed = set()
    if node:
        if relation == "imports":
            observed = {
                f"{item.module}.{alias.name}"
                for item in ast.walk(node)
                if isinstance(item, ast.ImportFrom) and item.module
                for alias in item.names
            }
        elif relation == "calls":
            observed = {ast.unparse(item.func) for item in ast.walk(node)
                        if isinstance(item, ast.Call)}
        elif isinstance(node, ast.ClassDef):
            observed = {ast.unparse(base) for base in node.bases}
    module = str(Path(path).with_suffix("")).replace("/", ".")
    AST_RELATIONS.append({
        "source": f"{module}.{symbol}" if symbol else module,
        "relation": relation, "target": target,
        "source_present": target in observed,
    })

# Verify the absent name independently before using an abstention question.
ABSENT_SYMBOL = "kilin_evaluation_absent_symbol_7f39"
ABSENT_CONFIRMED = not any(
    ABSENT_SYMBOL in (project.source / name).read_text(encoding="utf-8")
    for name in SOURCE_HASHES
)
print("Absent-symbol premise confirmed:", ABSENT_CONFIRMED)


def inspect_index(session):
    """Compare actual persisted graph/index data with source-derived facts.

    Parameters
    ----------
    session : RAGSession
        Normal model-backed session after initial indexing.

    Returns
    -------
    dict
        Deterministic coverage, definitions and selected static relations.
        Unresolved lexical calls are distinguished from qualified targets.

    """
    from kilin.retrievals import load_indexes

    graph = session.indexes.code_graph
    indexed = set(_source_paths(session.chunks))
    expected = set(INDEXABLE_PATHS)
    entities = {(entity.name, str(entity.path).replace("\\", "/"))
                for entity in graph.entities}
    edges = {(edge.source, edge.relation, edge.target)
             for edge in graph.relations}
    definitions = []
    for (path, symbol), fact in AST_FACTS.items():
        name = str(Path(path).with_suffix("")).replace("/", ".")
        definitions.append({
            "file": path, "symbol": symbol,
            "result": "stale" if fact is None else
            "pass" if (f"{name}.{symbol}", path) in entities else "fail",
        })
    relations = []
    for reference in AST_RELATIONS:
        source, kind, target = (
            reference["source"], reference["relation"], reference["target"]
        )
        candidates = [target]
        if kind == "calls" and target == "select_history":
            candidates.insert(0, "kilin.answering.select_history")
        found = next((value for value in candidates
                      if (source, kind, value) in edges), None)
        relations.append({
            **reference, "graph_target": found,
            "result": "stale" if not reference["source_present"]
            else "pass" if found else "fail",
            "resolution": "lexical" if found == "select_history"
            else "qualified/external" if found else "missing",
        })
    disk = load_indexes(project.root / "indexes" / "retrieval.json")
    report = {
        "copied_files": len(SOURCE_HASHES),
        "indexable_files": len(expected), "indexed_files": len(indexed),
        "missing_index_paths": sorted(expected - indexed),
        "unexpected_index_paths": sorted(indexed - expected),
        "chunks": len(session.chunks),
        "entities": len(graph.entities), "relations": len(graph.relations),
        "definitions": definitions, "relation_checks": relations,
        "disk_fingerprint_matches": (
            disk.fingerprint == session.indexes.fingerprint
        ),
        "disk_graph_matches": disk.code_graph == graph,
        "index_status": session.index_status,
    }
    print("Index coverage:", len(indexed), "/", len(expected))
    print("Missing source paths:", report["missing_index_paths"])
    print("Graph definitions:", [item["result"] for item in definitions])
    print("Static relations:", [item["result"] for item in relations])
    return report


kilin/rag.py:1231 RAGSession.ask
kilin/rag.py:675 RAGSession
kilin/conversations.py:331 ConversationStore.create
kilin/conversations.py:463 ConversationStore.rename
kilin/answering.py:811 route_question
kilin/answering.py:1287 select_history
kilin/workspace.py:329 prepare_sources_workspace
kilin/project.py:392 configure_project
kilin/project_type.py:1057 project_type
Absent-symbol premise confirmed: True


## 5. Direct routing and repository discovery

These standalone questions check inventory, named code definitions, the
retrieval of relevant implementation, import relationships, and exact
mentions. Paths such as `kilin/rag.py` are verified independently from the
answer text. A symbol name appearing in an answer is only a weak anchor.

The first populated session also checks actual indexed file coverage,
selected definitions, imports, calls and inheritance, plus consistency with
the persisted graph. Qualified and unresolved lexical call targets are
reported separately. These deterministic checks do not run a model judge.

In [12]:
with conversation("direct-routes") as ctx:
    INFRASTRUCTURE_RESULTS["index"] = inspect_index(ctx["session"])
    run_case(
        ctx, "D01", "Which source folders and file types are "
        "available in this code project?",
        intent="project_inventory", retrieval="none",
    )
    run_case(
        ctx, "D02", "Explain RAGSession.ask in kilin/rag.py "
        "and how it decides what evidence to retrieve.",
        intent="code_explanation", retrieval="search",
        paths=("kilin/rag.py",), answer_terms=("RAGSession",),
        definitions=(("kilin/rag.py", "RAGSession.ask"),),
    )
    run_case(
        ctx, "D03", "How does ConversationStore.rename in "
        "kilin/conversations.py persist a conversation title?",
        intent="code_explanation", retrieval="search",
        paths=("kilin/conversations.py",),
        answer_terms=("rename",),
        definitions=(("kilin/conversations.py",
                      "ConversationStore.rename"),),
    )
    run_case(
        ctx, "D04", "Which imports from kilin.answering does "
        "kilin/rag.py use? Explain their role.",
        intent="code_explanation", retrieval="search",
        paths=("kilin/rag.py",),
    )
    run_case(
        ctx, "D05", "List every exact occurrence of RAGSession "
        "in kilin/rag.py.",
        intent="mention_listing", retrieval="none",
        choice="first20",
    )
show_results(prefix="D")


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (146147 > 131072). Running this sequence through the model will result in indexing errors


Conversation direct-routes: conversation_2026-11-09T15-15-55.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit
Index coverage: 205 / 205
Missing source paths: []
Graph definitions: ['pass', 'pass', 'pass', 'pass', 'pass', 'pass', 'pass', 'pass', 'pass']
Static relations: ['pass', 'pass', 'pass']


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
project_inventory → project_inventory,project_inventory,need []; got [],none,0:,1158/14,5.10s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"3: rag.py, evidence_budget.py, repository.py",59265/618,515.98s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"3: conversations.py, conversation_format.py, c…",11534/753,82.30s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [2],search,0:,60142/8215,1612.13s,MATCH,incomplete,"scope, execution"


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [],none,1: rag.py (1 scanned),1728/33,6.06s,MATCH,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
D01,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,completed,---,0,---,5.10
D02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,3,---,515.98
D03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,3,---,82.30
D04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[2],MATCH,incomplete,REVIEW,0,execution,1612.13
D05,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,---,1 retained / 1 scanned,---,6.06


[{'id': 'D01',
  'mode': 'full',
  'label': 'direct-routes',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Which source folders and file types are available in this code project?',
  'expected_intent': 'project_inventory',
  'expected_tool': 'project_inventory',
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': 'none',
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': [],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding-

### 5.1 Saved-conversation and index reuse (opt-in)

`RUN_PERSISTENCE_CHECK` closes and reopens the existing direct-routing
conversation without adding a question. It compares committed answers and
representations, verifies the index fingerprint is unchanged, and records
whether the index was reused. No source files are edited and no generation
is requested, although the backend models still load during startup.
Incremental add/change/remove indexing is not exercised by this notebook.

In [13]:
if RUN_PERSISTENCE_CHECK:
    entry = CONVERSATION_INDEX["direct-routes"]
    trace_path = project_dir / entry["path"]
    before = json.loads(trace_path.read_text())["sessions"][
        "05_conversations"
    ]["resume_state"]["turns"]
    with conversation(
        "reopen-check", resume_from=entry["conversation_id"]
    ) as ctx:
        session = ctx["session"]
        expected = [(item["question"], item["answer"],
                     item["parameters"].get("representation"))
                    for item in before]
        actual = [(turn.question, turn.answer,
                   turn.parameters.get("representation"))
                  for turn in session.turns]
        result = {
            "history_matches": actual == expected,
            "index_reused": session.index_status == "reused",
            "fingerprint_matches": session.indexes.fingerprint
            == entry["effective_settings"]["index_fingerprint"],
            "turns": len(actual),
        }
        INFRASTRUCTURE_RESULTS["persistence"] = result
        print(result)
else:
    print("Conversation reload omitted (opt-in).")

Conversation reopen-check: conversation_2026-11-09T15-15-55.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit
{'history_matches': True, 'index_reused': True, 'fingerprint_matches': True, 'turns': 5}


## 6. Code referents and multi-turn continuity

These are real conversations with committed answers. The notebook checks
that source-specific references survive follow-up questions and explicit
switches between implementation files. Router related-turn IDs and the
actual history selected for generation are reported separately.


### 6.1 Source-file and symbol switches

An explicit module/function name establishes a preferred source scope. An
anaphoric follow-up should retain that source until another file or
symbol is introduced. We test source identity independently of intent.


In [14]:
with conversation("code-relations") as ctx:
    run_case(
        ctx, "A01", "Explain route_question in kilin/answering.py.",
        intent="code_explanation", standalone=True,
        paths=("kilin/answering.py",),
    )
    run_case(
        ctx, "A02", "How does that function validate the "
        "router's selected related turns?",
        intent="code_explanation", standalone=False,
        related_to=("A01",), paths=("kilin/answering.py",),
    )
    run_case(
        ctx, "A03", "Now explain prepare_sources_workspace "
        "in kilin/workspace.py instead.",
        intent="code_explanation", paths=("kilin/workspace.py",),
    )
    run_case(
        ctx, "A04", "Does this second function modify the "
        "original source directory or make a copy?",
        intent="code_explanation", standalone=False,
        related_to=("A03",), paths=("kilin/workspace.py",),
    )
    probe_route(
        ctx, "A05", "Return to route_question: how does it "
        "handle malformed router output?",
        intent="code_explanation", standalone=False,
        related_to=("A01",),
    )
    probe_route(
        ctx, "A06", "Compare those two functions' roles in "
        "the creation and answering pipelines.",
        intent="code_explanation", standalone=False,
    )
show_results(prefix="A")


Conversation code-relations: conversation_2026-11-09T16-50-34.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: answering.py, rag.py",13642/1305,125.96s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need [1]; got [],search,"5: scientific_paper.py, registry.py, rag.py, a…",6894/779,55.04s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"5: workspace.py, engine.py, project.py, worksp…",21435/832,153.68s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → file_reading,batch_review,need [3]; got [3],none,1: workspace.py,2814/670,30.08s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need [1]; got [],search,0:,1490/15,3.29s,REVIEW,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → project_inventory,---,"need []; got [2, 3, 4]",none,0:,1490/31,3.60s,REVIEW,probed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
A01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,125.96
A02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],REVIEW,completed,OK,5,---,55.04
A03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,5,---,153.68
A04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,file_reading,[3],REVIEW,completed,OK,1,---,30.08
A05,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],REVIEW,probed,---,---,---,3.29
A06,Qwen/Qwen3-8B-MLX-4bit,code_explanation,project_inventory,"[2, 3, 4]",REVIEW,probed,---,---,---,3.60


[{'id': 'A01',
  'mode': 'full',
  'label': 'code-relations',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Explain route_question in kilin/answering.py.',
  'expected_intent': 'code_explanation',
  'expected_tool': None,
  'expected_standalone': True,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': ['kilin/answering.py'],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ',
   'gen

### 6.2 Previously answered questions, follow-ups, and rephrasing

A request to rephrase the last answer is not a new code search. A question
about the last answer can often reuse its evidence. A new named symbol
should instead trigger the relevant code retrieval policy.


In [15]:
with conversation("code-continuity") as ctx:
    run_case(
        ctx, "C01", "Explain the role of ConversationStore in "
        "kilin/conversations.py.",
        intent="code_explanation", paths=("kilin/conversations.py",),
    )
    run_case(
        ctx, "C02", "What does its create method save?",
        intent="code_explanation", standalone=False,
        related_to=("C01",), paths=("kilin/conversations.py",),
    )
    run_case(
        ctx, "C03", "According to your previous answer, "
        "what information is recorded for the new conversation?",
        intent="conversation_followup", standalone=False,
        related_to=("C02",), retrieval="reuse",
    )
    run_case(
        ctx, "C04", "Rewrite that answer as two short bullets "
        "without adding any new facts.",
        intent="rephrase_previous", standalone=False,
        related_to=("C03",), retrieval="reuse",
    )
    probe_route(
        ctx, "C05", "Now explain configure_project in "
        "kilin/project.py.", intent="code_explanation",
    )
    probe_route(
        ctx, "C06", "What exactly did you say in the second bullet?",
        intent="conversation_followup", standalone=False,
        related_to=("C04",),
    )
show_results(prefix="C")


Conversation code-continuity: conversation_2026-11-09T16-56-51.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: conversations.py, conversation_format.py",10649/805,79.55s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need [1]; got [],search,"3: conversations.py, persistence.py, conversat…",6691/649,48.85s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
conversation_followup → fact_finding,find_fact,"need [2]; got [1, 2]",none,"3: conversations.py, README.md, engine.py",6264/6510,232.45s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
rephrase_previous → rephrase_previous,---,need [3]; got [3],reuse,"3: conversations.py, README.md, engine.py",3321/448,24.89s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,0:,1476/15,3.26s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
conversation_followup → conversation_followup,---,need [4]; got [4],reuse,0:,1476/23,3.42s,MATCH,probed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
C01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,79.55
C02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],REVIEW,completed,OK,3,---,48.85
C03,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,fact_finding,"[1, 2]",REVIEW,completed,---,3,---,232.45
C04,Qwen/Qwen3-8B-MLX-4bit,rephrase_previous,rephrase_previous,[3],MATCH,completed,---,3,---,24.89
C05,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,probed,---,---,---,3.26
C06,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[4],MATCH,probed,---,---,---,3.42


[{'id': 'C01',
  'mode': 'full',
  'label': 'code-continuity',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Explain the role of ConversationStore in kilin/conversations.py.',
  'expected_intent': 'code_explanation',
  'expected_tool': None,
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': ['kilin/conversations.py'],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding

## 7. Previous-result tools versus new searches

Exact mention listing, filtering the displayed subset, and showing the
source context of a cited occurrence are different operations. Neither
filtering nor source-context inspection should silently start an
unrestricted search. Routing-only probes examine the boundaries without
running expensive classification and semantic scanning.


In [16]:
with conversation("results-tools") as ctx:
    run_case(
        ctx, "S01", "List every exact occurrence of RAGSession "
        "in kilin/rag.py.",
        intent="mention_listing", retrieval="none",
        choice="first20",
    )
    run_case(
        ctx, "S02", "Show the surrounding source text of the "
        "first occurrence you just listed.",
        intent="source_context", standalone=False,
        related_to=("S01",), retrieval="none",
    )
    run_case(
        ctx, "S03", "Of those listed occurrences, keep only "
        "the ones in class or function definitions.",
        intent="refine_results", standalone=False,
        related_to=("S01",), retrieval="none",
    )
    probe_route(
        ctx, "S04", "Find new passages about how history "
        "selection works; do not just reuse that list.",
        intent="topic_listing", disallow=("source_context",),
    )
    probe_route(
        ctx, "S05", "Classify all exact uses of RAGSession "
        "as definitions, imports, or other references.",
        intent="mention_classification",
    )
    probe_route(
        ctx, "S06", "Filter only those earlier results for "
        "calls that construct a new RAGSession.",
        intent="refine_results", standalone=False,
        related_to=("S01",),
    )
show_results(prefix="S")


Conversation results-tools: conversation_2026-11-09T17-03-28.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [],none,1: rag.py (1 scanned),1394/33,4.35s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
source_context → source_context,show_source,need [1]; got [1],none,1: rag.py,1703/687,26.61s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
refine_results → mention_listing,scan_and_list,need [1]; got [1],none,"12: _model_completion.py, _project_creation.py…",1592/38,4.24s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
topic_listing → topic_listing,---,"need []; got [1, 2, 3]",none,0:,1461/28,3.41s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_classification → mention_classification,---,need []; got [1],none,0:,1463/22,3.29s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
refine_results → refine_results,---,"need [1]; got [1, 2, 3]",none,0:,1460/29,3.43s,MATCH,probed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
S01,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,---,1 retained / 1 scanned,---,4.35
S02,Qwen/Qwen3-8B-MLX-4bit,source_context,source_context,[1],MATCH,completed,---,1,---,26.61
S03,Qwen/Qwen3-8B-MLX-4bit,refine_results,mention_listing,[1],REVIEW,completed,---,12 retained / 205 scanned,---,4.24
S04,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,"[1, 2, 3]",MATCH,probed,---,---,---,3.41
S05,Qwen/Qwen3-8B-MLX-4bit,mention_classification,mention_classification,[1],MATCH,probed,---,---,---,3.29
S06,Qwen/Qwen3-8B-MLX-4bit,refine_results,refine_results,"[1, 2, 3]",MATCH,probed,---,---,---,3.43


[{'id': 'S01',
  'mode': 'full',
  'label': 'results-tools',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'List every exact occurrence of RAGSession in kilin/rag.py.',
  'expected_intent': 'mention_listing',
  'expected_tool': 'scan_and_list',
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': 'none',
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': [],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ',
   

### 7.1 Display, export and cancellation choices

These cases exercise GUI-equivalent callbacks without a graphical
interface. Saving an exhaustive list is opt-in. A cancelled
expensive operation should not be mistaken for a successful exhaustive
answer. All network confirmation requests are always cancelled.


In [17]:
with conversation("gui-paths") as ctx:
    run_case(
        ctx, "G01", "Which files are in the code project?",
        intent="project_inventory", choice="first20",
    )
    run_case(
        ctx, "G02", "List every exact mention of RAGSession "
        "throughout the code repository.",
        intent="mention_listing", choice="first20",
    )
    run_case(
        ctx, "G03", "Find every comment that discusses "
        "routing ambiguity rather than matching literal words.",
        intent="topic_listing", confirm="cancel",
        expected_outcome="cancelled",
    )
    if RUN_GUI_EXPORT_TEST:
        run_case(
            ctx, "G04", "List every exact mention of RAGSession "
            "throughout the code repository.",
            intent="mention_listing", choice="save_all",
        )
show_results(prefix="G")


Conversation gui-paths: conversation_2026-11-09T17-04-18.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
project_inventory → project_inventory,project_inventory,need []; got [],none,0:,1153/14,3.33s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [],none,"6: __init__.py, README.md, engine.py, app.js, …",1460/28,3.75s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
topic_listing → mention_listing,scan_and_list,need []; got [],none,0: (205 scanned),1545/27,3.91s,REVIEW,completed,execution


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [2],none,0: (205 scanned),1598/36,4.21s,MATCH,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
G01,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,completed,---,0,---,3.33
G02,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,---,6 retained / 205 scanned,---,3.75
G03,Qwen/Qwen3-8B-MLX-4bit,topic_listing,mention_listing,[],REVIEW,completed,---,0 retained / 205 scanned,execution,3.91
G04,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[2],MATCH,completed,---,0 retained / 205 scanned,---,4.21


[{'id': 'G01',
  'mode': 'full',
  'label': 'gui-paths',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Which files are in the code project?',
  'expected_intent': 'project_inventory',
  'expected_tool': 'project_inventory',
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': [],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ',
   'generation': {'temper

## 8. Routing boundaries (lightweight probes)

The following cases cover **all 14 code-project intents**, with explicit
negative contrasts between exact mentions, semantic passages, factual
extraction, whole-file reading, code explanation and code generation.
A probe invokes only the router; it never executes the selected tool.
Where multiple routes are defensible, use an exploratory unscored case
instead of declaring an arbitrary route correct.


In [18]:
with conversation("routing-boundaries") as ctx:
    run_case(
        ctx, "B00", "Explain RAGSession.ask in kilin/rag.py.",
        intent="code_explanation", paths=("kilin/rag.py",),
    )
    run_case(
        ctx, "B00b", "List exact occurrences of RAGSession "
        "in kilin/rag.py.", intent="mention_listing",
        choice="first20",
    )
    probe_route(ctx, "B01", "hello", intent="small_talk")
    probe_route(
        ctx, "B02", "Explain how RAGSession.ask chooses evidence.",
        intent="code_explanation",
    )
    probe_route(
        ctx, "B03", "What value does code.json specify for "
        "default_intent?", intent="fact_finding",
    )
    probe_route(
        ctx, "B04", "Write a new Python helper that returns "
        "the conversation names from a ConversationStore.",
        intent="code_generation",
    )
    probe_route(
        ctx, "B05", "Debug why a requested source filename "
        "might select a generated report with a similar name.",
        intent="debugging",
    )
    probe_route(
        ctx, "B06", "Read the entire kilin/gui/_paths.py file "
        "and review its implementation.", intent="file_reading",
    )
    probe_route(
        ctx, "B07", "How many notebooks are included in the "
        "project?", intent="project_inventory",
    )
    probe_route(
        ctx, "B08", "Find all exact uses of RAGSession.",
        intent="mention_listing",
    )
    probe_route(
        ctx, "B09", "Which files discuss conversation "
        "persistence by meaning?", intent="topic_listing",
    )
    probe_route(
        ctx, "B10", "Classify all exact occurrences of "
        "RAGSession as definitions or uses.",
        intent="mention_classification",
    )
    probe_route(
        ctx, "B11", "Of those listed instances, keep only "
        "the uses inside functions.",
        intent="refine_results", standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx, "B12", "Show the source context around the "
        "first occurrence you just listed.",
        intent="source_context", standalone=False,
        related_to=("B00b",),
    )
    probe_route(
        ctx, "B13", "What was your main point about "
        "RAGSession.ask?", intent="conversation_followup",
        standalone=False, related_to=("B00",),
    )
    probe_route(
        ctx, "B14", "Rewrite your explanation of RAGSession.ask "
        "in two sentences.", intent="rephrase_previous",
        standalone=False,
    )
    probe_route(
        ctx, "B15", "Which functions import code from "
        "kilin.answering?", intent="fact_finding",
    )
    probe_route(
        ctx, "B16", "Find the literal text 'code_graph' "
        "throughout the repository.", intent="mention_listing",
    )
    probe_route(
        ctx, "B17", "Find comments about source filtering "
        "without requiring an exact keyword.",
        intent="topic_listing",
    )
    probe_route(
        ctx, "B18", "Is kilin/project_types/code.json a "
        "different type of source from kilin/rag.py?",
        intent=None,
    )
    probe_route(
        ctx, "B19", "Explain why source hooks use a "
        "fallback for unrecognized file extensions.",
        intent="code_explanation",
    )
    probe_route(
        ctx, "B20", "Search kilin/project_types/code.json "
        "for the actual default intent setting.",
        intent="file_reading",
    )
    probe_route(
        ctx, "B21", "Add a new parser for JavaScript files.",
        intent="code_generation",
    )
    probe_route(
        ctx, "B22", "Which files handle generating a PDF report?",
        intent="topic_listing",
    )
    probe_route(
        ctx, "B23", "Tell me where KíLín makes the decision "
        "to ignore source files above the size limit.",
        intent="code_explanation",
    )
    probe_route(
        ctx, "B24", "Give me a full list of saved project "
        "profile settings.", intent="fact_finding",
    )
show_results(prefix="B")


Conversation routing-boundaries: conversation_2026-11-09T17-04-38.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: rag.py, repository.py",54969/590,460.71s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [1],none,1: rag.py (1 scanned),1538/41,4.63s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
small_talk → small_talk,---,need []; got [],none,0:,0/0,0.01s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [1],search,0:,1391/23,3.27s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
fact_finding → fact_finding,---,need []; got [],none,0:,1392/15,3.13s,MATCH,probed,---


... (19 outputs left out)

Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,0:,1401/15,3.16s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
fact_finding → project_inventory,---,need []; got [],none,0:,1393/14,3.14s,REVIEW,probed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
B00,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,460.71
B00b,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[1],MATCH,completed,---,1 retained / 1 scanned,---,4.63
B01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,probed,---,---,---,0.01
B02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[1],MATCH,probed,---,---,---,3.27
B03,Qwen/Qwen3-8B-MLX-4bit,fact_finding,fact_finding,[],MATCH,probed,---,---,---,3.13
B04,Qwen/Qwen3-8B-MLX-4bit,code_generation,code_generation,[],MATCH,probed,---,---,---,3.10
B05,Qwen/Qwen3-8B-MLX-4bit,debugging,debugging,[],MATCH,probed,---,---,---,3.32
B06,Qwen/Qwen3-8B-MLX-4bit,file_reading,file_reading,[],MATCH,probed,---,---,---,3.10
B07,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,probed,---,---,---,3.10
B08,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[2],MATCH,probed,---,---,---,3.27


[{'id': 'B00',
  'mode': 'full',
  'label': 'routing-boundaries',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Explain RAGSession.ask in kilin/rag.py.',
  'expected_intent': 'code_explanation',
  'expected_tool': None,
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': ['kilin/rag.py'],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community/Qwen3-Embedding-0.6B-4bit-DWQ',
   'generation'

## 9. Cross-format scope and source attribution

The development repository includes Python, JSON, JavaScript, HTML and
Jupyter notebooks. Each format is indexed through KíLín's normal text and
source extraction. The purpose here is **not** to compare parsers, but to
verify that answers use the specific files requested, and do not substitute
a similarly named output or an unrelated source.


In [19]:
with conversation("hard-scope") as ctx:
    run_case(
        ctx, "H01", "Explain how renderProjectTypes in "
        "kilin/gui/web/app.js builds the project-type choices.",
        intent="code_explanation", paths=("kilin/gui/web/app.js",),
    )
    run_case(
        ctx, "H02", "How does kilin/gui/web/index.html load "
        "the JavaScript application code?",
        intent="code_explanation",
        paths=("kilin/gui/web/index.html",),
    )
    run_case(
        ctx, "H03", "Explain the default intent and "
        "retrieval policy in kilin/project_types/code.json.",
        intent="code_explanation",
        paths=("kilin/project_types/code.json",),
    )
    run_case(
        ctx, "H04", "What does tutorial "
        "tutorials/150_logging_trace.ipynb demonstrate?",
        intent="code_explanation",
        paths=("tutorials/150_logging_trace.ipynb",),
    )
    probe_route(
        ctx, "H05", "Which source stores GUI default model choices?",
        intent="fact_finding",
    )
    probe_route(
        ctx, "H06", "Explain how the Python engine coordinates "
        "with the JavaScript frontend.", intent="code_explanation",
    )
    probe_route(
        ctx, "H07", "Show me the exact lines from "
        "index.html that include app.js.", intent="file_reading",
    )
show_results(prefix="H")


Conversation hard-scope: conversation_2026-11-09T17-13-42.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: app.js, project_type.py",82720/4301,1570.43s,MATCH,completed,execution


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → file_reading,batch_review,need []; got [],none,1: index.html,9790/583,56.98s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: code.json, project_type.py",6863/504,45.83s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → topic_listing,find_passages,need []; got [],none,1: 150_logging_trace.ipynb,6542/152,27.38s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
fact_finding → project_inventory,---,need []; got [],none,0:,1441/14,3.29s,REVIEW,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,0:,1444/15,3.47s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
file_reading → source_context,---,need []; got [2],none,0:,1446/22,3.63s,REVIEW,probed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
H01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,execution,1570.43
H02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,file_reading,[],REVIEW,completed,OK,1,---,56.98
H03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,45.83
H04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,topic_listing,[],REVIEW,completed,OK,1,---,27.38
H05,Qwen/Qwen3-8B-MLX-4bit,fact_finding,project_inventory,[],REVIEW,probed,---,---,---,3.29
H06,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,probed,---,---,---,3.47
H07,Qwen/Qwen3-8B-MLX-4bit,file_reading,source_context,[2],REVIEW,probed,---,---,---,3.63


[{'id': 'H01',
  'mode': 'full',
  'label': 'hard-scope',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Explain how renderProjectTypes in kilin/gui/web/app.js builds the project-type choices.',
  'expected_intent': 'code_explanation',
  'expected_tool': None,
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': ['kilin/gui/web/app.js'],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'answering_model': 'Qwen/Qwen3-8B-MLX-4bit',
   'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
   'embedding_model': 'mlx-community

### 9.1 Definition and code-graph ground-truth checks

These questions target definitions that were checked directly in the copied
snapshot using Python's AST. They also test multi-module source coverage.
A retrieved filename and a correct symbol name do not guarantee a correct
explanation of callers or control flow; assess those claims manually.

Selected explanation cases require the actual definition's start line to
occur in retrieved evidence. The definition is found independently in the
copied Python source; a missing definition skips only dependent tests.
The index/graph checks in Section 5 are distinct from these answer checks.

In [20]:
with conversation("code-ground-truth") as ctx:
    run_case(
        ctx, "X01", "Where is RAGSession.ask defined? Explain "
        "its important arguments and return value.",
        intent="code_explanation", paths=("kilin/rag.py",),
        answer_terms=("ask",),
        definitions=(("kilin/rag.py", "RAGSession.ask"),),
    )
    run_case(
        ctx, "X02", "How does ConversationStore.create "
        "manage conversation identity and optional branching?",
        intent="code_explanation",
        paths=("kilin/conversations.py",),
    )
    run_case(
        ctx, "X03", "Explain how kilin/rag.py uses "
        "kilin/answering.py to route questions. Cite both files.",
        intent="code_explanation", paths=(
            "kilin/rag.py", "kilin/answering.py",
        ),
    )
    run_case(
        ctx, "X04", "Explain how project_type in "
        "kilin/project_type.py loads the configured project type.",
        intent="code_explanation",
        paths=("kilin/project_type.py",),
    )
    run_case(
        ctx, "X05", "What role does code_graph play in "
        "the code project's capabilities configuration?",
        intent="code_explanation",
        paths=("kilin/project_types/code.json",),
    )
show_results(prefix="X")


Conversation code-ground-truth: conversation_2026-11-09T17-42-20.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"10: rag.py, _contract.py, literature.json, _de…",22764/1031,186.79s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"4: conversations.py, engine.py, conversation_f…",7085/898,65.83s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: answering.py, rag.py",66407/647,658.49s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: project_type.py, project.py",13574/665,91.11s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → project_inventory,project_inventory,need []; got [4],none,0:,1536/22,4.66s,REVIEW,completed,scope


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
X01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,10,---,186.79
X02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,4,---,65.83
X03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,658.49
X04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,91.11
X05,Qwen/Qwen3-8B-MLX-4bit,code_explanation,project_inventory,[4],REVIEW,completed,REVIEW,0,---,4.66


[{'id': 'X01',
  'mode': 'full',
  'label': 'code-ground-truth',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Where is RAGSession.ask defined? Explain its important arguments and return value.',
  'expected_intent': 'code_explanation',
  'expected_tool': None,
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': ['kilin/rag.py'],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': ['ask'],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [{'file': 'kilin/rag.py',
    'symbol': 'RAGSession.ask',
    'kind': 'FunctionDef',
    'line': 1231,
    'end_line': 2031}],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx',
   'ans

### 9.2 Duplicate symbols, imports and absent identifiers

The repository contains repeated `run` definitions and identical basenames
in different tool packages. Exact paths should preserve their identity.
The absent identifier is verified independently before the question is
asked; its response requires manual checking for invented APIs. No additional
synthetic source repository is created.

In [21]:
with conversation("symbol-boundaries") as ctx:
    run_case(
        ctx, "N01", "Explain run in kilin/tools/find_fact/tool.py. "
        "Distinguish it from run in kilin/tools/find_passages/tool.py.",
        intent="code_explanation",
        paths=("kilin/tools/find_fact/tool.py",
               "kilin/tools/find_passages/tool.py"),
        definitions=(("kilin/tools/find_fact/tool.py", "run"),
                     ("kilin/tools/find_passages/tool.py", "run")),
    )
    probe_route(
        ctx, "N02", "Now explain the other run function's role.",
        intent="code_explanation", standalone=False,
        related_to=("N01",),
    )
    question = f"Where is {ABSENT_SYMBOL} defined and what does it do?"
    if ABSENT_CONFIRMED:
        run_case(ctx, "N03", question, intent="code_explanation")
    else:
        skip_case("symbol-boundaries", "N03", question,
                  intent="code_explanation",
                  reason="The absent-name premise no longer holds.")
    run_case(
        ctx, "N04", "Explain why IncompleteModelResponse in "
        "kilin/_model_completion.py inherits RuntimeError.",
        intent="code_explanation", paths=("kilin/_model_completion.py",),
        definitions=(("kilin/_model_completion.py",
                      "IncompleteModelResponse"),),
    )
    run_case(
        ctx, "N05", "Explain how kilin/__init__.py exposes "
        "RAGSession to callers. Does that import depend on Qt GUI startup?",
        intent="code_explanation", paths=("kilin/__init__.py",),
    )
show_results(prefix="N")

Conversation symbol-boundaries: conversation_2026-11-09T17-59-14.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"3: tool.py, __init__.py, _registry.py",17553/804,126.34s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need [1]; got [1],search,0:,1270/23,3.03s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → mention_listing,scan_and_list,need []; got [],none,0: (205 scanned),1520/36,4.26s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: _model_completion.py, tool.py",3701/608,33.15s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"3: __init__.py, app.py, rag.py",5779/512,39.02s,MATCH,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
N01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,3,---,126.34
N02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[1],MATCH,probed,---,---,---,3.03
N03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,mention_listing,[],REVIEW,completed,---,0 retained / 205 scanned,---,4.26
N04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,33.15
N05,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,3,---,39.02


[{'id': 'N01',
  'mode': 'full',
  'label': 'symbol-boundaries',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Explain run in kilin/tools/find_fact/tool.py. Distinguish it from run in kilin/tools/find_passages/tool.py.',
  'expected_intent': 'code_explanation',
  'expected_tool': None,
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': ['kilin/tools/find_fact/tool.py',
   'kilin/tools/find_passages/tool.py'],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [{'file': 'kilin/tools/find_fact/tool.py',
    'symbol': 'run',
    'kind': 'FunctionDef',
    'line': 268,
    'end_line': 536},
   {'file': 'kilin/tools/find_passages/

## 10. Expensive end-to-end tools (opt-in)

Routing probes cover all 14 intents cheaply. The optional cases here
execute the expensive tools and distinguish a **Quick sample** from a
**Continue/full** scan. Do not interpret a Quick result as exhaustive.
Full semantic and fact-extraction scans may make many local model calls.


In [22]:
if RUN_LONG_TOOLS:
    with conversation("optional-review") as ctx:
        run_case(
            ctx, "L01", "Which files describe how the system "
            "reuses previous-turn evidence, by meaning rather "
            "than by exact word matching?",
            intent="topic_listing", confirm="continue",
        )
        run_case(
            ctx, "L02", "Give a complete list of profile setting "
            "names supported by kilin/project.py.",
            intent="fact_finding", confirm="continue",
        )
        run_case(
            ctx, "L03", "Classify all exact occurrences of "
            "RAGSession as definitions, imports, or uses.",
            intent="mention_classification", confirm="continue",
        )
    show_results(prefix="L")
else:
    print("Full model-backed scans omitted: RUN_LONG_TOOLS=False")

if RUN_FULL_SOURCE_READ:
    with conversation("optional-file-review") as ctx:
        run_case(
            ctx, "F01", "Read all of kilin/gui/_paths.py and "
            "review its interface and behavior.",
            intent="file_reading", confirm="continue",
        )
    show_results(prefix="F")
else:
    print("Whole-file review omitted: RUN_FULL_SOURCE_READ=False")


Conversation optional-review: conversation_2026-11-09T18-02-45.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
topic_listing → topic_listing,find_passages,need []; got [],none,"60: answering.py, chunking.py, conversation_fo…",505661/5865,2120.13s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
fact_finding → mention_listing,scan_and_list,need []; got [],none,0: (1 scanned),1524/40,9.01s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_classification → mention_classification,batch_classify,need []; got [1],none,"13: rag.py, __init__.py, engine.py, trace_logg…",7252/1499,83.82s,MATCH,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
L01,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,completed,---,60,---,2120.13
L02,Qwen/Qwen3-8B-MLX-4bit,fact_finding,mention_listing,[],REVIEW,completed,---,0 retained / 1 scanned,---,9.01
L03,Qwen/Qwen3-8B-MLX-4bit,mention_classification,mention_classification,[1],MATCH,completed,---,13 retained / 205 scanned,---,83.82


Conversation optional-file-review: conversation_2026-11-09T18-39-44.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
file_reading → file_reading,batch_review,need []; got [],none,1: _paths.py,5643/1031,54.60s,MATCH,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
F01,Qwen/Qwen3-8B-MLX-4bit,file_reading,file_reading,[],MATCH,completed,---,1,---,54.60


## 11. Debugging and code generation

These tests ask KíLín to inspect existing code and propose changes **as
text only**. The notebook does not apply generated patches, execute model-
written programs, change source files, or invoke an unsafe interpreter.
For correctness, inspect the cited functions and verify generated code
independently before using it in a real repository.

The generated function must occur in exactly one Python-labelled or
unlabelled fenced block. That block is parsed and compiled, with its index
recorded. Signature and expected API-call checks are structural checks only;
a valid AST cannot establish that the implementation behaves correctly.
No generated source is imported or executed.

In [23]:
with conversation("code-generation") as ctx:
    run_case(
        ctx, "J01", "Investigate how a named source file is "
        "resolved by kilin/tools/_shared/choose_document.py. "
        "What could make a similarly named generated output "
        "appear instead of the requested source? Give checks "
        "to confirm the behavior before suggesting a fix.",
        intent="debugging",
    )
    run_case(
        ctx, "J02", "Write a small standalone Python function "
        "list_conversation_names(store) that reads the names "
        "from ConversationStore.conversations(). Follow "
        "KíLín's documentation style and 79-character limit. "
        "Return the function as text, not a patch.",
        intent="code_generation", answer_terms=("def ",),
        generated_function="list_conversation_names",
        generated_parameters=("store",),
        generated_calls=("store.conversations",),
    )
    probe_route(
        ctx, "J03", "Modify the JSON project type configuration "
        "to enable source suffixes .js and .html.",
        intent="code_generation",
    )
    if RUN_CODE_REVIEW:
        run_case(
            ctx, "J04", "Review the logic that rejects source "
            "files exceeding a profile size limit. Identify "
            "a concrete bug if supported, otherwise explain "
            "how to test it; do not invent a bug.",
            intent="debugging",
        )
show_results(prefix="J")


Conversation code-generation: conversation_2026-11-09T18-40-44.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
debugging → code_explanation,---,need []; got [],search,"2: choose_document.py, tool.py",5994/909,56.49s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_generation → code_generation,---,need []; got [],search,"2: conversations.py, engine.py",5603/1274,70.22s,MATCH,completed,content


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_generation → project_inventory,---,need []; got [],none,0:,1430/14,3.59s,REVIEW,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
debugging → code_explanation,---,need []; got [],search,"4: conversations.py, project_type.py, rag.py, …",5322/636,42.84s,REVIEW,completed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
J01,Qwen/Qwen3-8B-MLX-4bit,debugging,code_explanation,[],REVIEW,completed,---,2,---,56.49
J02,Qwen/Qwen3-8B-MLX-4bit,code_generation,code_generation,[],MATCH,completed,---,2,content,70.22
J03,Qwen/Qwen3-8B-MLX-4bit,code_generation,project_inventory,[],REVIEW,probed,---,---,---,3.59
J04,Qwen/Qwen3-8B-MLX-4bit,debugging,code_explanation,[],REVIEW,completed,---,4,---,42.84


[{'id': 'J01',
  'mode': 'full',
  'label': 'code-generation',
  'model': 'Qwen/Qwen3-8B-MLX-4bit',
  'router_model': 'Qwen/Qwen3-4B-MLX-4bit',
  'question': 'Investigate how a named source file is resolved by kilin/tools/_shared/choose_document.py. What could make a similarly named generated output appear instead of the requested source? Give checks to confirm the behavior before suggesting a fix.',
  'expected_intent': 'debugging',
  'expected_tool': None,
  'expected_standalone': None,
  'expected_related_ids': [],
  'forbidden_related_ids': [],
  'expected_retrieval': None,
  'expected_sources': [],
  'forbidden_sources': [],
  'expected_paths': [],
  'forbidden_paths': [],
  'disallowed_intents': [],
  'expected_answer_terms': [],
  'expected_generated_function': None,
  'expected_generated_parameters': None,
  'expected_generated_calls': [],
  'expected_definitions': [],
  'expected_outcome': 'completed',
  'confirmation_policy': 'quick',
  'effective_settings': {'backend': 'mlx'

## 12. Controlled answering-model comparison (opt-in)

Seven tasks per model: a three-turn source-specific explanation sequence,
then independent JSON fact extraction, debugging, Python generation and
HTML/JavaScript interaction questions. The four independent tasks use fresh
conversations. The continuity follow-ups run only after their prerequisite
matches the expected task, is completed, contains the required definition or
source, and has a nonempty final answer. These guards do not prove semantic
correctness; review the answer separately.

Every skipped model or dependent case is exported with a reason. Startup
failures are separate from query failures; cleanup cannot hide the original
exception. The factual tool may make several internal calls, and its
`continue` confirmation permits normal processing. This is an end-to-end
comparison, not an identical-prompt decoder benchmark.

In [24]:
ANSWER_COMPARISON = (
    {
        "question": "Explain RAGSession.ask in kilin/rag.py and "
                    "describe how it retrieves evidence.",
        "intent": "code_explanation", "standalone": True,
        "paths": ("kilin/rag.py",), "answer_terms": ("RAGSession",),
        "definitions": (("kilin/rag.py", "RAGSession.ask"),),
    },
    {
        "question": "Which parameters does that method accept, "
                    "and what do they control?",
        "intent": "code_explanation", "standalone": False,
        "paths": ("kilin/rag.py",),
    },
    {
        "question": "Compare that method with ConversationStore.create "
                    "in kilin/conversations.py.",
        "intent": "code_explanation", "standalone": False,
        "paths": ("kilin/rag.py", "kilin/conversations.py"),
    },
    {
        "question": "What exact default_intent value is configured in "
                    "kilin/project_types/code.json?",
        "intent": "fact_finding", "confirm": "continue",
        "paths": ("kilin/project_types/code.json",),
        "answer_terms": (json.loads((project.source /
            "kilin/project_types/code.json").read_text())["default_intent"],),
    },
    {
        "question": "Diagnose how kilin/rag.py restores the answering "
                    "session when use_chat_model fails. Identify an "
                    "actual failure mode or state what remains unproven.",
        "intent": "debugging", "paths": ("kilin/rag.py",),
    },
    {
        "question": "Write a standalone function "
                    "list_conversation_names(store) returning names from "
                    "ConversationStore.conversations(). Use a Python "
                    "code block, documented style and 79 columns.",
        "intent": "code_generation", "generated_function":
            "list_conversation_names", "generated_parameters": ("store",),
        "generated_calls": ("store.conversations",),
        "paths": ("kilin/conversations.py",),
    },
    {
        "question": "Explain how kilin/gui/web/app.js and "
                    "kilin/gui/web/index.html connect the user interface. "
                    "Distinguish visible markup from event-handling code.",
        "intent": "code_explanation",
        "paths": ("kilin/gui/web/app.js", "kilin/gui/web/index.html"),
    },
)

if RUN_MODEL_COMPARISON:
    for model_key in dict.fromkeys(COMPARISON_MODELS):
        # Continuity is end-to-end: later history is model-specific.
        group = f"answer-model-{model_key}"
        with conversation(group, model=model_key, optional=True) as ctx:
            model_available = ctx is not None
            previous = None
            for number, spec in enumerate(ANSWER_COMPARISON[:3], 1):
                case_id = f"M_{model_key}_{number:02d}"
                reason = (
                    "Model setup failed; see startup records." if ctx is None
                    else "Prerequisite answer failed task/scope checks."
                    if number > 1 and not _seed_is_valid(previous) else None
                )
                if reason:
                    skip_case(group, case_id, spec["question"],
                              intent=spec["intent"], model=model_key,
                              reason=reason)
                    continue
                options = dict(spec)
                if number > 1:
                    options["related_to"] = (previous["id"],)
                previous = run_case(ctx, case_id, **options)
        # Independent tasks get fresh conversations and identical questions.
        for number, spec in enumerate(ANSWER_COMPARISON[3:], 4):
            group = f"answer-model-{model_key}-independent-{number}"
            case_id = f"M_{model_key}_{number:02d}"
            if not model_available:
                skip_case(group, case_id, spec["question"],
                          intent=spec["intent"], model=model_key,
                          reason="Earlier model initialization failed.")
                continue
            with conversation(group, model=model_key, optional=True) as ctx:
                model_available = ctx is not None
                if ctx is None:
                    skip_case(group, case_id, spec["question"],
                              intent=spec["intent"], model=model_key,
                              reason="Model setup failed; see startup log.")
                else:
                    run_case(ctx, case_id, **spec)
    show_answer_model_comparison()
else:
    print("Answering-model comparison omitted (opt-in).")


Conversation answer-model-qwen8b: conversation_2026-11-09T18-43-42.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: rag.py, evidence_budget.py",55003/463,454.02s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → fact_finding,find_fact,need [1]; got [1],none,"7: chat.py, embeddings.py, engine.py, rag.py, …",21199/14269,559.85s,REVIEW,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → -,---,need []; got [],---,0:,-,0.00s,SKIPPED,skipped,---


Conversation answer-model-qwen8b-independent-4: conversation_2026-11-09T19-00-41.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
fact_finding → mention_listing,scan_and_list,need []; got [],none,1: code.json (1 scanned),1394/33,4.72s,REVIEW,completed,---


... (15 outputs left out)

Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_generation → code_generation,---,need []; got [],search,"3: conversations.py, engine.py, context_expans…",5663/6960,878.03s,MATCH,completed,---


Conversation answer-model-qwen38_27b-independent-7: conversation_2026-11-10T00-04-48.json
  Answering: lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → -,---,need []; got [],---,0:,-,949.62s,ERROR,failed,---


Prompt,Answer model,Intent,Sources,Seconds,Recorded summary,Route,Execution
01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,"rag.py, evidence_budget.py",454.02,The `RAGSession.ask` method in `kilin/rag.py` is a central function that orchestrates the process of answering a question by retrieving evidence…,MATCH,completed
01,lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly,code_explanation,"rag.py, evidence_budget.py",1947.33,"`RAGSession.ask` is the public one-turn orchestration method for a loaded KíLín project: it retrieves evidence, generates an answer, and saves t…",MATCH,completed
02,Qwen/Qwen3-8B-MLX-4bit,fact_finding,"chat.py, embeddings.py, engine.py, rag.py, context_expansion.py, sources_py.py, sources.py",559.85,"The question refers to ""that method,"" but no specific method is named.",REVIEW,completed
02,lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly,fact_finding,,11769.18,This operation could not be completed.,REVIEW,incomplete
03,Qwen/Qwen3-8B-MLX-4bit,-,,0.00,(summary unavailable),SKIPPED,skipped
03,lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly,-,,0.00,(summary unavailable),SKIPPED,skipped
04,Qwen/Qwen3-8B-MLX-4bit,mention_listing,code.json,4.72,"Found **1 mention** of “default_intent” in **code.json** (case-sensitive, whole words).",REVIEW,completed
04,lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly,mention_listing,code.json,6.37,"Found **1 mention** of “default_intent” in **code.json** (case-sensitive, whole words).",REVIEW,completed
05,Qwen/Qwen3-8B-MLX-4bit,debugging,rag.py,436.87,The `RAGSession.use_chat_model` function in `kilin/rag.py` attempts to switch the answering model while preserving the conversation state.,MATCH,completed
05,lukaskremla/Qwen3.8-27B-4bit-MLX-TextOnly,debugging,,2108.48,This operation could not be completed.,MATCH,incomplete


## 13. Controlled router-model comparison (opt-in)

A validated explanation and exact-mention list form one shared seed.
Each router receives a branch with identical saved questions and answers;
branch restoration is checked before probing. Eighteen probes retain the
contextual comparisons and cover all 14 configured code intents.
Routing probes do not add answers or execute the chosen tools, so the seed
history stays fixed. Failed seeds, unavailable models or differing branches
produce explicit skipped records. Router settings remain core-owned;
the notebook records actual models rather than changing router policies.


In [25]:
ROUTER_PROBES = (
    # Preserve four contextual comparison prompts, then cover all intents.
    ("What does its rename method persist?", "code_explanation", "Rseed00"),
    ("List exact mentions of RAGSession.", "mention_listing", None),
    ("Find files about GUI and backend communication.", "topic_listing", None),
    ("How does that class handle creating branches?", "code_explanation",
     "Rseed00"),
    ("Hello", "small_talk", None),
    ("Explain route_question in kilin/answering.py.",
     "code_explanation", None),
    ("What value is default_intent in kilin/project_types/code.json?",
     "fact_finding", None),
    ("Write a new helper to list saved conversation names.",
     "code_generation", None),
    ("Debug a source-selection error that returns the wrong named file.",
     "debugging", None),
    ("Read all of kilin/gui/_paths.py and review its implementation.",
     "file_reading", None),
    ("How many source files are indexed?", "project_inventory", None),
    ("List all exact RAGSession mentions.", "mention_listing", None),
    ("Find passages about caching by meaning.", "topic_listing", None),
    ("Classify all exact RAGSession mentions as definitions or calls.",
     "mention_classification", None),
    ("Filter the occurrences you listed to uses inside functions.",
     "refine_results", "Rseed01"),
    ("Show context for the first occurrence you listed.",
     "source_context", "Rseed01"),
    ("What was your main point about ConversationStore?",
     "conversation_followup", "Rseed00"),
    ("Rewrite your explanation of ConversationStore in two sentences.",
     "rephrase_previous", "Rseed00"),
)
assert {item[1] for item in ROUTER_PROBES} == set(code_intents())

if RUN_ROUTER_COMPARISON:
    parent_id = None
    seed_history = []
    with conversation("router-seed", optional=True) as seed_ctx:
        specs = (
            ("Rseed00", "Explain ConversationStore in kilin/conversations.py.",
             "code_explanation"),
            ("Rseed01", "List exact mentions of ConversationStore in "
             "kilin/conversations.py.", "mention_listing"),
        )
        seeds = []
        for case_id, question, intent in specs:
            if seed_ctx is None:
                seeds.append(skip_case(
                    "router-seed", case_id, question, intent=intent,
                    reason="Router-seed initialization failed.",
                ))
            else:
                seeds.append(run_case(
                    seed_ctx, case_id, question, intent=intent,
                    paths=("kilin/conversations.py",), choice="first20",
                ))
        if seed_ctx is not None and all(_seed_is_valid(x) for x in seeds):
            parent_id = seed_ctx["info"].conversation_id
            seed_history = [(t.question, t.answer)
                            for t in seed_ctx["session"].turns]
    for index, router in enumerate(COMPARISON_ROUTERS, 1):
        group = f"router-model-{index}"
        if parent_id is None:
            for number, (question, intent, _) in enumerate(ROUTER_PROBES, 1):
                skip_case(group, f"R{index}_{number:02d}", question,
                          intent=intent, mode="route-only", router=router,
                          reason="Shared seed did not meet its contract.")
            continue
        with conversation(group, router=router, branch_from=parent_id,
                          optional=True) as ctx:
            history_ok = ctx is not None and [
                (turn.question, turn.answer) for turn in ctx["session"].turns
            ] == seed_history
            if ctx is not None:
                CONVERSATION_INDEX[group]["branch_matches_seed"] = history_ok
                ctx["case_turn_ids"].update(Rseed00=1, Rseed01=2)
            reason = "Branch unavailable or history differs."
            if history_ok:
                started = perf_counter()
                try:
                    # Load once before the matrix; do not repeatedly retry
                    # an unavailable router for every probe.
                    ctx["session"]._router_chat()
                except Exception as error:
                    history_ok = False
                    reason = f"Router unavailable: {type(error).__name__}: "
                    reason += str(error)
                    STARTUP_RECORDS.append({
                        "label": group, "router": router,
                        "status": "failed", "error": reason,
                    })
                CONVERSATION_INDEX[group]["router_startup_seconds"] = (
                    perf_counter() - started
                )
            for number, (question, intent, related) in enumerate(
                ROUTER_PROBES, 1
            ):
                case_id = f"R{index}_{number:02d}"
                if not history_ok:
                    skip_case(group, case_id, question, intent=intent,
                              mode="route-only", router=router,
                              reason=reason)
                else:
                    probe_route(
                        ctx, case_id, question, intent=intent,
                        related_to=(related,) if related else (),
                        standalone=False if related else None,
                    )
    show_results(prefix="R")
else:
    print("Router-model comparison omitted (opt-in).")


Conversation router-seed: conversation_2026-11-10T00-20-49.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,need []; got [],search,"2: conversations.py, conversation_format.py",10640/818,80.13s,MATCH,completed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,scan_and_list,need []; got [],none,1: conversations.py (1 scanned),1513/32,3.83s,MATCH,completed,---


Conversation router-model-1: conversation_2026-11-10T00-22-19.json
  Answering: Qwen/Qwen3-8B-MLX-4bit
  Router: Qwen/Qwen3-4B-MLX-4bit


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
code_explanation → code_explanation,---,"need [1]; got [1, 2]",search,0:,1334/26,3.40s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
mention_listing → mention_listing,---,need []; got [],none,0:,1335/14,2.88s,MATCH,probed,---


... (32 outputs left out)

Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
source_context → source_context,---,need [2]; got [2],none,0:,1336/22,5.45s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
conversation_followup → conversation_followup,---,need [1]; got [1],reuse,0:,1336/23,5.48s,MATCH,probed,---


Expected → actual,Tool,Prior links,Retrieval,Evidence,Tokens in/out,Time,Routing check,Execution,Other checks
rephrase_previous → rephrase_previous,---,need [1]; got [1],reuse,0:,1337/23,5.48s,MATCH,probed,---


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
Rseed00,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,80.13
Rseed01,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,OK,1 retained / 1 scanned,---,3.83
R1_01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,"[1, 2]",MATCH,probed,---,---,---,3.40
R1_02,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,probed,---,---,---,2.88
R1_03,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,[],MATCH,probed,---,---,---,3.08
R1_04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,"[1, 2]",MATCH,probed,---,---,---,3.10
R1_05,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,probed,---,---,---,0.00
R1_06,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,probed,---,---,---,2.89
R1_07,Qwen/Qwen3-8B-MLX-4bit,fact_finding,fact_finding,[],MATCH,probed,---,---,---,2.90
R1_08,Qwen/Qwen3-8B-MLX-4bit,code_generation,code_generation,"[1, 2]",MATCH,probed,---,---,---,3.09


## 14. Review and export

Review routing, evidence scope, execution, content anchors and trace
consistency **independently**. The saved checkpoint is reconciled against
the notebook's full answer, chosen intent, history, source paths and KíLín
recorded summary. Probes do not have committed answers to reconcile.

The reference table shows AST-derived definition locations from the copied
snapshot. It is intentionally not a model-generated evaluation score.

Read `Completed` as completion of the expected selected operation, not as a
correct answer. Full attempts, probes, partial work, cancellation and skips
are distinct. Inspect the manifest's infrastructure checks for graph/index
coverage, stale AST references and optional reload results.

In [26]:
reconciliation = verify_saved_workspace()
coverage = show_coverage()
diagnostics = show_diagnostic_totals()
code_references = show_code_references()
all_end_to_end = show_results(mode="full")
all_routing = show_results(mode="route-only")
print("Recorded cases:", len(CASE_RESULTS))
print("Conversations:", len(CONVERSATION_INDEX))


Committed turns verified: 59
Notebook/workspace discrepancies: 0


Intent,Tool,Full attempts,Probes,Selected,Completed,Partial,Cancelled,Failed,Skipped
code_explanation,---,31,14,23,22,0,0,3,2
fact_finding,find_fact,3,6,0,0,0,0,0,0
code_generation,---,3,5,3,3,0,0,0,0
debugging,---,4,3,2,1,0,0,1,0
file_reading,batch_review,1,5,1,1,0,0,0,0
project_inventory,project_inventory,3,3,3,3,0,0,0,0
mention_listing,scan_and_list,7,6,7,7,0,0,0,0
topic_listing,find_passages,2,8,1,1,0,0,0,0
mention_classification,batch_classify,1,4,1,1,0,0,0,0
refine_results,filter_results,1,4,0,0,0,0,0,0


Mode,Cases,Match,Review,---,Errors,Skipped,Scope flags,Citation flags,Trace flags,Anchor flags
full,62,42,17,0,1,2,4,2,0,1
route-only,72,63,8,1,0,0,0,0,0,0


Case,Reference file,Symbol,Snapshot fact,In evidence,Manual quality
D02,kilin/rag.py,RAGSession.ask,FunctionDef at line 1231,yes,pending
D03,kilin/conversations.py,ConversationStore.rename,FunctionDef at line 463,yes,pending
A01,kilin/answering.py,route_question,FunctionDef at line 811,yes,pending
A03,kilin/workspace.py,prepare_sources_workspace,FunctionDef at line 329,yes,pending
H01,kilin/gui/web/app.js,(file),whole file,yes,pending
H02,kilin/gui/web/index.html,(file),whole file,yes,pending
H03,kilin/project_types/code.json,(file),whole file,yes,pending
H04,tutorials/150_logging_trace.ipynb,(file),whole file,yes,pending
X01,kilin/rag.py,RAGSession.ask,FunctionDef at line 1231,yes,pending
X02,kilin/conversations.py,ConversationStore.create,FunctionDef at line 331,yes,pending


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
E01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,completed,---,0,---,1.93
E02,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,completed,---,0,---,3.24
E03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,---,0,execution,18.95
E04,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,---,0 retained / 0 scanned,---,4.07
D01,Qwen/Qwen3-8B-MLX-4bit,project_inventory,project_inventory,[],MATCH,completed,---,0,---,5.10
D02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,3,---,515.98
D03,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,3,---,82.30
D04,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[2],MATCH,incomplete,REVIEW,0,execution,1612.13
D05,Qwen/Qwen3-8B-MLX-4bit,mention_listing,mention_listing,[],MATCH,completed,---,1 retained / 1 scanned,---,6.06
A01,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,completed,OK,2,---,125.96


Case,Model,Expected,Actual,Links,Route,Execution,Scope,Evidence files,Warnings,Seconds
A05,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],REVIEW,probed,---,---,---,3.29
A06,Qwen/Qwen3-8B-MLX-4bit,code_explanation,project_inventory,"[2, 3, 4]",REVIEW,probed,---,---,---,3.60
C05,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[],MATCH,probed,---,---,---,3.26
C06,Qwen/Qwen3-8B-MLX-4bit,conversation_followup,conversation_followup,[4],MATCH,probed,---,---,---,3.42
S04,Qwen/Qwen3-8B-MLX-4bit,topic_listing,topic_listing,"[1, 2, 3]",MATCH,probed,---,---,---,3.41
S05,Qwen/Qwen3-8B-MLX-4bit,mention_classification,mention_classification,[1],MATCH,probed,---,---,---,3.29
S06,Qwen/Qwen3-8B-MLX-4bit,refine_results,refine_results,"[1, 2, 3]",MATCH,probed,---,---,---,3.43
B01,Qwen/Qwen3-8B-MLX-4bit,small_talk,small_talk,[],MATCH,probed,---,---,---,0.01
B02,Qwen/Qwen3-8B-MLX-4bit,code_explanation,code_explanation,[1],MATCH,probed,---,---,---,3.27
B03,Qwen/Qwen3-8B-MLX-4bit,fact_finding,fact_finding,[],MATCH,probed,---,---,---,3.13


Recorded cases: 134
Conversations: 27


### 14.1 Inspect failures and annotate code quality

A routing `MATCH` is a routing-only claim. A correct filename can still
support an incorrect implementation description, and valid syntax may
still be unsuitable code. Review source-grounded cases individually and
record a qualitative verdict with `annotate_case`, for example:

```python
annotate_case(
    "D02", quality="mixed",
    note="Retrieved rag.py, but misidentified the history-selection step.",
)
```

False-positive citation warnings can occur when source text contains
literal numbered references. Verify attribution before classifying a
citation as fabricated.


In [27]:
needs_review = [
    item for item in CASE_RESULTS
    if item["status"] in ("REVIEW", "ERROR")
]
for item in needs_review:
    print(
        f"{item['id']:<17} {item['status']:<7} "
        f"{', '.join(item.get('issues', []))}"
    )
print("Routing reviews and errors:", len(needs_review))
other = [
    item for item in CASE_RESULTS
    if any(
        item.get("checks", {}).get(name)
        for name in ("scope", "execution", "trace", "content")
    )
]
print("Separate scope/evidence/trace flags:", len(other))


A02               REVIEW  routing: standalone, routing: missing prior-turn link
A04               REVIEW  routing: intent, routing: tool
A05               REVIEW  routing: standalone, routing: missing prior-turn link
A06               REVIEW  routing: intent
C02               REVIEW  routing: standalone, routing: missing prior-turn link
C03               REVIEW  routing: intent, routing: tool, routing: retrieval policy
S03               REVIEW  routing: intent, routing: tool
G03               REVIEW  routing: intent, routing: tool, execution: expected cancelled; observed completed
B15               REVIEW  routing: intent
B17               REVIEW  routing: intent
B24               REVIEW  routing: intent
H02               REVIEW  routing: intent, routing: tool
H04               REVIEW  routing: intent, routing: tool
H05               REVIEW  routing: intent
H07               REVIEW  routing: intent
X05               REVIEW  routing: intent, routing: tool, scope: missing source path: ki

### 14.2 Save the evaluation

Exports are stored **inside this experiment workspace**:

- `cases.json`: complete raw answers, recorded summaries, routing decisions,
  source paths, tools, usage and trace diagnostics;
- `cases.csv`: compact comparisons, distinct scan/evidence/call counters;
- `manifest.json`: model settings, Git version/status, imported file hashes,
  expected intents, AST definitions, conversation names and enabled flags.

Source code, notebook outputs and model reasoning may be present in the
workspace and should be treated as private development material.


In [28]:
evaluation_dir = export_evaluation()


Exported: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_code_routing_20261109-161523/outputs/notebook_evaluation


## 15. Interpreting the results

Diagnose discrepancies by layer before changing KíLín:

1. **Routing:** wrong intent, wrong tool or wrong preceding-turn link.
2. **Retrieval scope:** named module missing, stale symbol resolution,
   unrelated evidence, or insufficient cross-file coverage.
3. **Code facts:** compare against AST definitions, source lines, imports
   and actual implementation—not against plausible descriptions.
4. **Tool execution:** distinguish Quick samples, Continue/full scans,
   cancellation, incomplete generation and tool-internal calls.
5. **Conversational fidelity:** confirm that a prior answer succeeded
   before interpreting pronouns and model comparisons.
6. **Trace integrity:** compare original saved answers and KíLín-authored
   summaries against committed conversation state.

The benchmark follows the **development checkout**, not a frozen release.
The snapshot hashes and recorded Git state make every run auditable.
Change one KíLín component at a time and use the same scenarios to verify
whether a particular failure improves without weakening existing behavior.
